# EP08｜梯度下降與邏輯迴歸：第一個會學習的模型

[![在 Colab 開啟](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chainchen1103/llm-from-scratch/blob/main/notebooks/ep08/ep08_gradient_descent.ipynb)

這份 notebook 對應網站上的 [EP08 圖文教學](https://chainchen1103.github.io/llm-from-scratch/episodes/ep08/)，段落和網站一樣，可以邊讀邊跑。EP07 的最後，我們讓玩具模型的分數一步一步換成 $z - 1.0 \times (q - \hat{p})$（$q$ 是模型的機率，$\hat{p}$ 是資料裡每個 token 的比例），大約 20 步，損失就降到地板。往負梯度走損失會下降，EP04 就知道了；可是還有三件事沒回答：**一步該走多遠？**（EP04 走了大約 0.12，EP06 每步走 0.1 倍，EP07 走 1.0 倍，這些數怎麼來的？）**損失長什麼樣子、谷底只有一個嗎？模型怎麼從資料學？**（玩具模型只是把機率調成資料的比例；真正的模型要看線索猜答案。）

這集用四個工具回答：**梯度下降、學習率、邏輯迴歸、損失地形**，接起來就是本集重點：**第一個會學習的模型**。它只有兩個參數、只分「冷」「熱」兩類，可是「算預測 → 算損失 → 算梯度 → 走一步」這個迴圈，就是訓練 LLM 的骨架。

1. 工具一：梯度下降
2. 工具二：學習率（含四個學習率、又長又窄的碗兩張圖）
3. 工具三：邏輯迴歸（示範資料、標準化、sigmoid、二元交叉熵、梯度 p − y）
4. 工具四：損失地形（含損失地形的圖）
5. **本集重點：第一個會學習的模型**（訓練過程的圖在第 6 節動手做畫）
6. 動手做：從零寫出 `make_weather_data`、`gradient_descent`、`sigmoid`、`bce_loss`、`bce_grad`、`fit_logistic`、`accuracy` 並用檢查函式測試；寫出畫訓練過程的 `weather_data`、`train`、`draw_loss`、`draw_fit`；和 PyTorch 對照
7. 弄壞它：學習率太小、太大；正負號寫反；梯度用加總、不用平均；不標準化（含圖）；資料完全分得開（含圖）

最後是參考解答。

這份 notebook 只用到 NumPy 和 matplotlib，Colab 已經內建，不用安裝任何東西；前幾集寫好的 `softmax`（EP01）、`logsumexp`、`log_softmax`（EP02）、`numerical_gradient`（EP04）、`cross_entropy_loss`（EP07）也已經定義在下面，不用匯入。在自己的電腦上，用 EP01 建立的 `llm-labs` 專案開啟即可。動手做的最後一個檢查和「和 PyTorch 對照」那一格會和 PyTorch 比對：Colab 已經內建 PyTorch；沒裝 PyTorch 的話（EP01 步驟 7），那些地方會自動跳過。

**使用方式**：從上到下依序執行（Shift + Enter）。弄壞它會用到第 6 節動手做寫的函式，還沒寫完的話那幾格只會印出提示，不會中斷；弄壞它 4、5 有幾格要走 10 萬步，各要跑幾秒。弄壞它的每一格，先在心裡猜結果再執行。參考解答在最後，先自己寫寫看。

In [ ]:
# Colab 已經內建 NumPy 和 matplotlib，不用安裝；在自己的電腦上，請用 EP01 建立的 llm-labs 環境。
# %pip install -q numpy matplotlib

import math
import sys

import matplotlib
import matplotlib.pyplot as plt
import numpy as np

print("Python", sys.version.split()[0], "| NumPy", np.__version__, "| matplotlib", matplotlib.__version__)

這份 notebook 用 NumPy 2.5 測試過。程式格裡的註解是說明，印出來的結果就在格子下方，和網站上一樣。圖上的文字用英文，因為 matplotlib 預設的字型沒有中文，會變成方框。

下面是前幾集動手做寫好的函式，這集會用到：EP01 的 `softmax`（每個分數都加同一個數，機率不變）；EP02 的 `logsumexp`（$\ln \sum_k e^{z_k}$，不會溢位）和 `log_softmax`（log 機率＝分數 − logsumexp）；EP04 的 `numerical_gradient`（每一格各做一次中央差分）；EP07 的 `cross_entropy_loss`（平均交叉熵，nat）。最後的顏色和 `tidy` 是畫圖用的小工具：淡灰色格線、拿掉上面和右邊的框。

還會用到這幾個前提：EP03 的線性層 $Wx + b$（權重、偏差），`np.linalg.norm` 是向量的長度；EP04 的碗 $x^2 + 3y^2$ 在 $(1, 2)$ 的梯度是 $(2x, 6y) = (2, 12)$，負梯度是下降最快的方向；EP05 的鏈鎖律；EP07：損失＝平均交叉熵（nat），壓低它就是最大概似估計，對分數的梯度是 $q - y$。

**寫法約定**：$\theta$（念 theta）是全部參數排成的一排數字，$\eta$（念 eta）是學習率，$\nabla f(\theta)$ 是 $f$ 在 $\theta$ 的梯度（EP04，$\nabla$ 念 nabla），$\leftarrow$ 念「換成」。**工具一、工具二的 $x$、$y$ 是碗的兩個座標，也就是要調的參數；從工具三開始，$x$ 是（標準化後的）氣溫、$y$ 是答案（1 = 冷、0 = 熱）**，意思不一樣。$p$ 是模型給「冷」的機率（工具一、二程式裡的 `p` 是碗上的位置）；一天的損失寫成 $\ell$（小寫的 L），平均起來寫成 $\text{loss}$（同 EP07）。$\ln$ 是自然對數（`np.log`），單位 nat。

In [ ]:
def softmax(z, axis=-1):                  # EP01
    """沿著 axis 把分數 z 轉成機率。z 可以是任意形狀。"""
    z = np.asarray(z, dtype=float)
    shifted = z - z.max(axis=axis, keepdims=True)   # 每一列減掉自己的最大值
    e = np.exp(shifted)
    return e / e.sum(axis=axis, keepdims=True)      # 每一列除以自己的總和


def logsumexp(z, axis=-1, keepdims=False):    # EP02
    """數值穩定地算 log(Σ exp(z))，沿著 axis 做。z 可以是任意形狀。
    回傳形狀和 z.sum(axis=axis, keepdims=keepdims) 一樣：
    z 形狀 (4, 7)、axis=-1 → (4,)；keepdims=True → (4, 1)。"""
    z = np.asarray(z, dtype=float)
    m = z.max(axis=axis, keepdims=True)                              # 每一列自己的最大值
    out = m + np.log(np.sum(np.exp(z - m), axis=axis, keepdims=True))
    if not keepdims:
        out = np.squeeze(out, axis=axis)                             # 拿掉長度 1 的那個維度
    return out


def log_softmax(z, axis=-1):              # EP02
    """回傳 log(softmax(z))，但不能先算 softmax 再取 log。輸出形狀和 z 一樣。"""
    z = np.asarray(z, dtype=float)
    return z - logsumexp(z, axis=axis, keepdims=True)                # keepdims=True 才能廣播回每一列


def numerical_gradient(f, x, h=1e-5):     # EP04
    """f 吃一個 NumPy 陣列、吐一個數。對 x 的每一格各做一次中央差分，
    回傳和 x 形狀一樣的陣列（每一格是 f 對那一格的偏導數）。不能改到呼叫端的 x。"""
    x = np.array(x, dtype=float)          # 複製一份、轉成小數：不改到呼叫端，整數輸入也能動一點點
    grad = np.zeros_like(x)
    for idx in np.ndindex(x.shape):       # (0,)、(1,)…；二維時是 (0, 0)、(0, 1)…
        old = x[idx]
        x[idx] = old + h
        f_plus = f(x)
        x[idx] = old - h
        f_minus = f(x)
        x[idx] = old                      # 復原：直接放回原值，不靠 +h、-h 抵消
        grad[idx] = (f_plus - f_minus) / (2 * h)
    return grad


def _check_targets(logits, targets):      # EP07：cross_entropy_loss 用的檢查
    """檢查 logits 是 (N, K)、targets 是長度 N、0 到 K − 1 的整數；不對就 raise ValueError。"""
    logits = np.asarray(logits, dtype=float)
    targets = np.asarray(targets)
    if logits.ndim != 2:
        raise ValueError(f"logits 要是 (N, K)，收到 {logits.shape}")
    n, k = logits.shape
    if targets.shape != (n,):
        raise ValueError(f"targets 要是長度 {n} 的一維陣列，收到 {targets.shape}")
    if not np.issubdtype(targets.dtype, np.integer):
        raise ValueError(f"targets 要是整數，收到 {targets.dtype}")
    if np.any(targets < 0) or np.any(targets >= k):
        raise ValueError(f"targets 要在 0 到 {k - 1} 之間：{targets}")
    return logits, targets


def cross_entropy_loss(logits, targets):  # EP07
    """logits：(N, K) 的分數；targets：長度 N 的整數（每個位置正確 token 的 ID）。回傳平均交叉熵
    （nat，float），和 PyTorch 的 F.cross_entropy 一樣。logits 不是二維、targets 長度不對、
    不是整數、或不在 0 到 K − 1，raise ValueError。"""
    logits, targets = _check_targets(logits, targets)
    n = logits.shape[0]
    return float(-np.mean(log_softmax(logits, axis=-1)[np.arange(n), targets]))


SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]   # 畫圖用：藍、橘、綠、黃（依序給每條線）
INK, MUTED = "#1E2A44", "#5B6577"                       # 文字、淡一點的文字


def tidy(ax, grid=True):                  # 畫圖用：淡灰色格線、拿掉上面和右邊的框
    if grid:
        ax.grid(True, color="0.88", linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ["top", "right"]:
        ax.spines[side].set_visible(False)

## 1. 工具一：梯度下降

### 把 EP04 那一步一直走下去

EP04 站在碗 $f(x, y) = x^2 + 3y^2$ 的 $(1, 2)$，算出梯度 $(2, 12)$，往負梯度走了**一步**。走到新的位置，坡的方向就變了，所以在那裡重新算梯度，再走一步；一直這樣走，叫**梯度下降 gradient descent**：

$$
\theta \leftarrow \theta - \eta\,\nabla f(\theta)
$$

白話：站在哪裡，就在那裡算梯度，往反方向走「$\eta$ 倍的梯度」，再算、再走。

$f$ 是要變小的函數，訓練時就是損失。$\eta$ 叫**學習率 learning rate**，是人選的、不是學出來的，所以也是超參數（EP00 的溫度也是）。

$\eta = 0.1$ 時，$x$ 換成 $x - 0.1 \times 2x = 0.8x$，$y$ 換成 $y - 0.1 \times 6y = 0.4y$：每一步 $x$ 乘 0.8、$y$ 乘 0.4。手算兩步：$(1, 2) \to (0.8, 0.8) \to (0.64, 0.32)$，高度 $13 \to 0.64 + 1.92 = 2.56 \to 0.4096 + 0.3072 = 0.7168$。

每一行依序是步數、位置、高度、梯度（學習率 0.1）：

In [ ]:
def bowl(p):
    return p[0] ** 2 + 3 * p[1] ** 2          # EP04 的碗 f(x, y) = x² + 3y²

def bowl_grad(p):
    return np.array([2 * p[0], 6 * p[1]])     # 梯度 (2x, 6y)

p = np.array([1.0, 2.0])
for step in range(4):
    g = bowl_grad(p)
    print(step, p, bowl(p), g)
    p = p - 0.1 * g                           # 往負梯度走一步：學習率 0.1

和手算一樣（`2.559999999999999` 是捨入誤差，EP02）。梯度也一步一步變短：越走，坡越平。

### 什麼時候停、每步走多遠

兩種停法：**走固定的步數**（上面的迴圈走 4 步）；或**梯度夠小就停**：梯度的長度小於一個門檻，代表這裡幾乎是平的。碗裡幾乎平的地方就是谷底附近（例外：弄壞它 5 的地形越走越平，卻沒有谷底）。

第一行是走了幾步、停在哪裡、高度；第二行是前五步走的距離，最後一個數是 EP04 那一步走的距離：

In [ ]:
p, step, lengths = np.array([1.0, 2.0]), 0, []
while np.linalg.norm(bowl_grad(p)) >= 1e-6:   # 梯度夠小（坡夠平）就停
    lengths.append(0.1 * np.linalg.norm(bowl_grad(p)))   # 實驗紀錄：這一步走的距離
    p = p - 0.1 * bowl_grad(p)
    step += 1
print(step, p, bowl(p))
print(np.round(lengths[:5], 4), 0.01 * np.linalg.norm(bowl_grad([1.0, 2.0])))   # 最後一個：EP04 那一步

- 66 步之後梯度短於 $10^{-6}$，高度 $1.6 \times 10^{-13}$：永遠不會剛好是 0，但越來越靠近谷底，這叫**收斂 converge**。$y$ 每步乘 0.4，早就到了（$10^{-26}$），後面幾十步都在等每步只乘 0.8 的 $x$。訓練 LLM 時通常是走固定的步數（看能花多少算力）。
- 第二行是前五步走的距離。學習率不是「走多遠」：**走多遠＝學習率 × 梯度的長度**。EP04 那一步 `p - 0.01 * grad` 就是學習率 0.01 的第一步，「大約 0.12」是 $0.01 \times$ 梯度的長度 12.17（最後一個數）。
- 學習率固定 0.1，前五步走的距離卻從 1.22 縮到 0.09，之後還會更小：**坡越陡走越多，越接近谷底走越少**，不用自己把步伐變小（EP15 的學習率排程 learning rate schedule 另有理由）。

## 2. 工具二：學習率

### 四個學習率

同一個碗、同一個起點，換四個學習率，各走 30 步。每一列依序是：學習率、$x$ 每步乘的數 $1 - 2\eta$、$y$ 每步乘的數 $1 - 6\eta$、前四步的 $y$、第 30 步的高度：

In [ ]:
def run(grad_fn, start, lr, steps):
    p, path = np.array(start), [np.array(start)]
    for _ in range(steps):
        p = p - lr * grad_fn(p)
        path.append(p)
    return np.array(path)                     # (steps + 1, 2)：每一步的位置

rates = [0.01, 0.1, 0.3, 0.34]
paths = [run(bowl_grad, [1.0, 2.0], lr, 30) for lr in rates]
for lr, path in zip(rates, paths):
    print(lr, round(1 - 2 * lr, 2), round(1 - 6 * lr, 2), np.round(path[:4, 1], 3), f"{bowl(path[30]):.3g}")

- **0.01：太小**。$y$ 每步只乘 0.94，30 步後高度還有 0.591。不會出錯，只是慢（EP04 故意用它：步伐小，線性近似才準）。
- **0.1：順利**。30 步降到 $1.5 \times 10^{-6}$。
- **0.3：偏大**。$y$ 乘 $-0.8$：每一步都跳到谷底的另一邊（2 → −1.6 → 1.28），越跳越近，還是會收斂。其實它和 0.1 一樣快：最慢的方向都是每步乘 0.8（下一格的圖右邊，兩條線後段一樣斜）。第 30 步的高度差 12 倍，是因為 0.1 最後剩下 $x^2$（從 1 出發），0.3 剩下 $3y^2$（從 $3 \times 2^2 = 12$ 出發）。
- **0.34：太大**。$y$ 乘 $-1.04$：每跳一次就離谷底更遠，30 步後高度 126，比起點的 13 還高。越走越遠叫**發散 diverge**（和高中數列的「發散」同一個字）。

畫成圖：左邊四張是每個學習率在碗上的前 12 步（灰線是等高線，黑點是起點 $(1, 2)$），右邊是每一步的高度（對數刻度）：

In [ ]:
fig = plt.figure(figsize=(12, 5.2))
gs = fig.add_gridspec(2, 4, width_ratios=[1, 1, 0.15, 2.3], hspace=0.35, wspace=0.25)   # 左邊 2 × 2 張小圖、右邊一張
gb = np.linspace(-3.6, 3.6, 361)
GX, GY = np.meshgrid(gb, gb)
for k, (lr, path, color) in enumerate(zip(rates, paths, SERIES)):
    ax = fig.add_subplot(gs[k // 2, k % 2])
    ax.contour(GX, GY, bowl([GX, GY]), levels=[0.5, 2, 5, 13, 25], colors="0.8", linewidths=0.8)   # 碗的等高線
    ax.plot(path[:13, 0], path[:13, 1], "o-", color=color, markersize=3.5, linewidth=1.5)          # 前 12 步
    ax.plot([1], [2], "o", color=INK, markersize=5, zorder=5)                                    # 起點 (1, 2)
    ax.set_title(f"lr {lr:g}: x × {1 - 2 * lr:g}, y × {1 - 6 * lr:g}", fontsize=9.5)
    ax.set(xlim=(-2.2, 2.2), ylim=(-3.5, 3.5), aspect="equal")
    ax.tick_params(labelsize=8)
    tidy(ax, grid=False)
fig.text(0.27, 0.99, "first 12 steps on the bowl x² + 3y², start (1, 2)", ha="center", va="top",
         fontsize=11, color=INK)

ax = fig.add_subplot(gs[:, 3])
for lr, path, color in zip(rates, paths, SERIES):
    heights = [bowl(q) for q in path]                                          # 每一步的高度
    ax.semilogy(np.arange(len(path)), heights, "o-", color=color, markersize=3, linewidth=1.8, label=f"lr {lr:g}")
    ax.annotate(f"{lr:g}: {heights[-1]:.3g}", (len(path) - 1, heights[-1]), xytext=(6, 0),
                textcoords="offset points", va="center", fontsize=9, color=INK)
ax.axhline(13, color=MUTED, linestyle=":", linewidth=1)
ax.text(20, 11, "start height 13", color=MUTED, fontsize=9, va="top")
ax.set(xlim=(0, 37), xlabel="step", ylabel="height f(x, y) (log scale)", title="height after each step")
tidy(ax)
ax.legend(loc="lower left", frameon=False)
plt.show()

0.01 慢慢滑、0.1 很快到底、0.3 在 $y = 0$ 兩邊來回跳著靠近、0.34 越跳越開；右圖 0.01 降得慢，0.1 和 0.3 後段一樣斜，0.34 往上。

### 為什麼上限是 1/3

$y$ 方向每一步：

$$
y \leftarrow y - \eta \times 6y = (1 - 6\eta)\,y
$$

白話：不管 $y$ 現在是多少，每一步都乘上同一個數 $1 - 6\eta$。

每步乘同一個數 $m$（高中的等比數列）：$|m| < 1$ 越乘越小，收斂，$m < 0$ 時每步換邊；$|m| > 1$ 越乘越大，發散。$|1 - 6\eta| < 1$ 就是 $0 < \eta < \frac{1}{3}$；$x$ 方向的 $|1 - 2\eta| < 1$ 只要 $\eta < 1$。兩個都要成立，所以 **$\eta < \frac{1}{3}$：最窄的方向決定上限**（剛好 $\frac{1}{3}$ 時，$y$ 永遠在 2 和 −2 之間跳）。

6 和 2 是什麼？$y$ 方向的斜率是 $6y$：$y$ 每多 1，斜率就多 6；$x$ 方向的斜率 $2x$ 每次只多 2。這個「斜率變化的快慢」叫**彎曲程度 curvature**：越大，碗在那個方向越窄。彎曲程度 $c$ 的方向每步乘 $1 - c\,\eta$，要 $|1 - c\,\eta| < 1$：

$$
\eta < \frac{2}{\text{最大的彎曲程度}}
$$

白話：學習率要小於「2 除以最彎的那個方向的彎曲程度」，這裡是 $\frac{2}{6} = \frac{1}{3}$。

真正的損失每個地方的彎曲程度都不一樣，算不出固定的上限，學習率通常是試出來的（弄壞它 1）。

### 又長又窄的碗：鋸齒狀前進

把碗的 $x$ 方向放平 10 倍：$0.1x^2 + 3y^2$，彎曲程度變成 0.2 和 6。上限還是 $\frac{1}{3}$（$y$ 決定）；用同樣的 0.3，$y$ 每步還是乘 $-0.8$，$x$ 卻只乘 $1 - 0.2 \times 0.3 = 0.94$。從 $(10, 1)$ 出發（高度一樣是 13）：第一行是起點的高度，和學習率 0.3 第 1、2、3、20 步的位置；第二行是學習率 0.1 第 20 步的位置：

In [ ]:
def flat_bowl(p):
    return 0.1 * p[0] ** 2 + 3 * p[1] ** 2    # x 方向放平 10 倍：又長又窄

def flat_grad(p):
    return np.array([0.2 * p[0], 6 * p[1]])

path = run(flat_grad, [10.0, 1.0], 0.3, 20)
print(flat_bowl(path[0]), np.round(path[[1, 2, 3, 20]], 3).tolist())
print(np.round(run(flat_grad, [10.0, 1.0], 0.1, 20)[20], 3))   # 學習率 0.1，第 20 步

$y$ 在谷底兩邊來回跳，20 步後只剩 0.012；$x$ 每步只挪一點點，20 步後還在 2.9。路徑是**鋸齒狀**的：橫越窄的方向來回跳，沿著長的方向慢慢爬。學習率 0.1 不會鋸齒（$y$ 乘 0.4，不換邊），$x$ 卻更慢，20 步後還在 6.68。想讓 $x$ 快一點，學習率又不能再大（$y$ 會發散）：**最窄的方向決定上限，最長的方向決定要走多久**。長和窄差得越多，走得越慢，工具三的「標準化」就是為了避免這件事。

畫出兩條路徑（星號是谷底）：

In [ ]:
path_zig = run(flat_grad, [10.0, 1.0], 0.3, 20)               # 學習率 0.3：鋸齒
path_slow = run(flat_grad, [10.0, 1.0], 0.1, 20)              # 學習率 0.1：不鋸齒，但更慢
fig, ax = plt.subplots(figsize=(10, 3.4))
GX, GY = np.meshgrid(np.linspace(-1, 11.5, 400), np.linspace(-1.4, 1.4, 200))
ax.contour(GX, GY, flat_bowl([GX, GY]), levels=[0.5, 1.5, 3, 6, 9.6, 13], colors="0.75", linewidths=0.9)
ax.plot(path_zig[:, 0], path_zig[:, 1], "o-", color=SERIES[0], markersize=4, linewidth=1.6, label="lr 0.3: zigzag")
ax.plot(path_slow[:, 0], path_slow[:, 1], "o-", color=SERIES[1], markersize=4, linewidth=1.6,
        label="lr 0.1: no zigzag, slower")
ax.annotate(f"step 20: x = {path_zig[20, 0]:.2f}", path_zig[20], xytext=(-20, -26), textcoords="offset points",
            color=INK, fontsize=9)
ax.annotate(f"step 20: x = {path_slow[20, 0]:.2f}", path_slow[20], xytext=(-10, -62), textcoords="offset points",
            color=INK, fontsize=9, arrowprops={"arrowstyle": "-", "color": MUTED, "linewidth": 0.8})
ax.plot([0], [0], "*", color=INK, markersize=12)                                          # 谷底 (0, 0)
ax.annotate("bottom", (0, 0), xytext=(-10, -18), textcoords="offset points", color=INK, fontsize=9)
ax.set(xlim=(-1, 11.5), ylim=(-1.4, 1.4), aspect="equal", xlabel="x", ylabel="y",
       title="long, narrow bowl 0.1x² + 3y², start (10, 1), 20 steps")
tidy(ax, grid=False)
ax.legend(loc="upper center", frameon=False, ncol=2, bbox_to_anchor=(0.5, -0.22))
plt.show()

學習率 0.3 鋸齒狀前進，停在 $x = 2.90$；0.1 不鋸齒但更慢，停在 $x = 6.68$。

（網站上有互動元件：在碗 $x^2 + 3y^2$ 的等高線上，從 $(1, 2)$ 出發換學習率（0.01、0.1、0.25、0.3、0.34，也可以用滑桿），看路徑、$x$ 和 $y$ 每步乘的 $1 - 2\eta$、$1 - 6\eta$ 和高度的對數圖；也可以切換成又長又窄的碗 $0.1x^2 + 3y^2$ 看鋸齒。）

### 補充：EP06、EP07 的玩具模型也一樣

把 EP07 那一步 $z - 1.0 \times (q - \hat{p})$ 的 1.0 換成別的學習率，看要幾步 KL 才小於 $10^{-6}$（走到 1000 步就不再走）。每一行是學習率和步數：

In [ ]:
p_hat = np.array([0.5, 0.2, 0.3])             # EP06 的資料比例：冷、熱、好
for lr in [1.0, 0.1, 3.0, 5.0, 8.0]:
    z_toy, n_steps = np.zeros(3), 0
    while n_steps < 1000:
        q = softmax(z_toy)
        if np.sum(p_hat * np.log(p_hat / q)) < 1e-6:   # KL(p̂ ‖ q) 夠小就停
            break
        z_toy = z_toy - lr * (q - p_hat)               # EP07 那一步，1.0 換成 lr
        n_steps += 1
    print(lr, n_steps)

1.0（就是 EP06 的 0.1）要 19 步，也就是 EP07 的「大約 20 步」（EP06 走了 50 步，其實 19 步就夠了）；換成 0.1（EP06 的 0.01）要 206 步，3.0 只要 4 步，5.0 反而要 81 步，8.0 走了 1000 步還停不下來。太小慢、太大停不下來，最好的在中間：EP06、EP07 選的數沒有特別的道理。能用 KL 決定何時停，是因為知道地板（熵）；真正訓練時不知道，就用工具一的兩種停法。

## 3. 工具三：邏輯迴歸

### 示範資料：「今天天氣很 ___」

EP00 的模型看了「今天天氣很」，給「冷」29.3%、「熱」22.9%。這集做一個極簡版：**只有「冷」「熱」兩個選項，只看一個線索：當天的氣溫**。下面是**示範資料**（不是真的天氣紀錄），照一個固定的規則產生：體感溫度＝氣溫加上 −6 到 6 度的隨機誤差，體感低於 20 度就寫「冷」。

依序印出 40 天的氣溫、答案（1 = 冷、0 = 熱），「冷」的天數、最熱的「冷」、最冷的「熱」，以及氣溫的平均和標準差：

In [ ]:
rng = np.random.default_rng(0)                # 示範資料：固定 seed 0
temp = rng.integers(5, 36, size=40)           # 40 天的氣溫（°C），5 到 35 度的整數（36 不含）
felt = temp + rng.uniform(-6, 6, size=40)     # 體感：每天多或少，最多 6 度
y = (felt < 20).astype(int)                   # 體感低於 20 度就寫「冷」：1 = 冷、0 = 熱
print(temp)
print(y)
print(y.sum(), temp[y == 1].max(), temp[y == 0].min())   # 冷的天數、最熱的「冷」、最冷的「熱」
print(temp.mean(), temp.std())                # 平均、標準差
x = (temp - 20) / 10                          # 標準化：20 度變 0，每 10 度算 1

- 40 天裡 21 天「冷」、19 天「熱」。19 度以下全是「冷」（17 天），25 度以上全是「熱」（14 天），**20 到 24 度兩種都有**（「冷」4 天、「熱」5 天）。同樣的氣溫，答案不一定一樣，就像同樣的前文，下一個 token 不一定一樣（EP07 的熵）。
- 從線索猜答案是哪一類，叫**分類 classification**；只有兩類叫**二元分類 binary classification**。氣溫這種拿來猜的數叫**特徵 feature**，答案 $y$ 叫**標籤 label**。LLM 猜下一個 token 也是分類，只是有 151,936 類。

### 標準化：把特徵縮放到差不多的範圍

最後一行把氣溫換成 $x = \frac{\text{氣溫} - 20}{10}$：20 度變 0、5 度變 −1.5、35 度變 1.5。把特徵平移、縮放到 0 附近、大小差不多 1，叫**標準化 standardization**。常見的做法是減掉平均、除以**標準差 standard deviation**（資料離平均大約多遠，`np.std`）；這份資料的平均 19.75、標準差 9.31，取整數 20 和 10 差不多，換算回氣溫也方便。

為什麼要做？等一下的模型有兩個參數 $w$、$b$。直接用 5 到 35 度，特徵又大又全是正的（$w$ 動一點，40 天的分數都往同一邊移，和 $b$ 動很像），損失就成了工具二那種又長又窄、還斜著放的碗（弄壞它 4）。減掉 20，特徵有正有負；再除以 10，大小差不多 1，碗就比較圓。

### 模型：sigmoid 就是兩個類別的 Softmax

先給「冷」打一個分數 $z = wx + b$：EP03 的線性層，只有 1 個輸入、1 個輸出，$w$ 是權重、$b$ 是偏差。分數要換成機率：讓「冷」的分數是 $z$、「熱」的分數固定是 0，做 Softmax：

$$
p = \frac{e^{z}}{e^{z} + e^{0}} = \frac{1}{1 + e^{-z}} = \sigma(z)
$$

白話：「冷」的機率，就是兩個分數 $[z, 0]$ 做 Softmax 之後的第一格（中間那一步是上下同除以 $e^z$）。

這個函數叫 **sigmoid**，記作 $\sigma$（念 sigma），也叫 logistic 函數。「熱」的分數可以固定是 0，因為 Softmax 只看分數的差（EP01：每個分數都加同一個數，機率不變），兩個類別只要一個數就夠了。

第一行是 sigmoid 在 $z = -5, -2, 0, 2, 5$ 的值；第二行左邊是其中一列分數 $[-2, 0]$，右邊是每一列做 Softmax 之後「冷」那一格：

In [ ]:
def sigmoid_naive(z):
    return 1 / (1 + np.exp(-z))               # 照定義寫（等一下會看到它的問題）

zs = np.array([-5.0, -2.0, 0.0, 2.0, 5.0])
scores = np.stack([zs, np.zeros(5)], axis=1)  # 兩排並成兩欄 (5, 2)：每一列是「冷」的分數 z、「熱」的分數 0
print(np.round(sigmoid_naive(zs), 4))
print(scores[1], np.round(softmax(scores)[:, 0], 4))   # 兩個類別的 Softmax，取「冷」那一格

- $\sigma(0) = 0.5$；$z$ 越大越靠近 1、越小越靠近 0，畫出來是一條 **S 形曲線**。$\sigma(-z) = 1 - \sigma(z)$ 是「熱」的機率（$0.1192 + 0.8808 = 1$）。斜率是 $\sigma(z)\,(1 - \sigma(z))$（EP07 Softmax Jacobian 的對角線 $q_i(1 - q_i)$），在 $z = 0$ 最大，是 0.25，往兩邊越來越平。
- 模型 $p = \sigma(wx + b)$ 是「冷」的機率。$w < 0$ 時，越冷（$x$ 越小）分數越大，越可能是「冷」；$|w|$ 越大，S 形越陡，模型越有把握。
- $p = 0.5$ 的地方是 $z = 0$，也就是 $x = -\frac{b}{w}$，換回氣溫（$20 + 10x$）是 $20 - \frac{10b}{w}$ 度：$w < 0$ 時，比它冷就判「冷」，比它熱就判「熱」。這個分界叫**決策邊界 decision boundary**，這集叫它**分界氣溫**。
- 這個模型叫**邏輯迴歸 logistic regression**：名字來自 logistic 函數，「迴歸」是歷史留下來的名字，它其實是拿來分類的。

（網站上有互動元件：拖動分數 $z$（−6 到 6），左邊是 sigmoid 曲線上的點，右邊是分數 $[z, 0]$ 做 Softmax 之後的兩根長條「冷」「熱」，第一根和點同高，就是 $\sigma(z)$。）

### 損失：二元交叉熵

每一天，模型的分佈是 $(p, 1 - p)$，正確答案的 one-hot 是 $(y, 1 - y)$。EP07 的交叉熵只剩兩項：

$$
\ell = -\bigl[\,y \ln p + (1 - y) \ln(1 - p)\,\bigr]
$$

白話：「冷」的日子是 $-\ln p$，「熱」的日子是 $-\ln(1 - p)$，都是「正確答案拿到的機率取 $-\ln$」（EP07 的驚訝程度）。

這叫**二元交叉熵 BCE binary cross-entropy**。40 天平均起來就是損失 $\text{loss}(w, b)$；壓低它就是最大概似估計（EP07）。

穩定的寫法：$p = \frac{e^z}{1 + e^z}$、$1 - p = \frac{1}{1 + e^z}$，所以 $\ln p = z - \ln(1 + e^z)$、$\ln(1 - p) = -\ln(1 + e^z)$，代進去：

$$
\ell = \ln(1 + e^{z}) - y\,z
$$

白話：直接從分數 $z$ 算損失，不用先算出機率。

$\ln(1 + e^z) = \ln(e^0 + e^z)$ 就是兩個數 $[0, z]$ 的 logsumexp（EP02），NumPy 寫成 `np.logaddexp(0, z)`，內部會先減掉最大值，不會溢位。

第一行依序是前五天「冷」的機率、前五天的損失、40 天的平均；第二、三行是穩定的寫法和 EP07 的 `cross_entropy_loss`；最後一行是 $w = b = 0$（什麼都沒學）：

In [ ]:
w, b = -3.0, 0.5                              # 隨便挑的一組參數
z = w * x + b                                 # 40 天的分數
p = sigmoid_naive(z)                          # 40 天「冷」的機率
per_day = -(y * np.log(p) + (1 - y) * np.log(1 - p))   # 每一天的二元交叉熵
print(np.round(p[:5], 3), np.round(per_day[:5], 4), per_day.mean())
print(np.mean(np.logaddexp(0, z) - y * z))   # 穩定的寫法：ln(1 + e^z) − y z
print(cross_entropy_loss(np.stack([z, np.zeros(40)], axis=1), 1 - y))   # EP07：分數 [z, 0]，「冷」是第 0 類
print(cross_entropy_loss(np.zeros((40, 2)), 1 - y))                       # w = b = 0：什麼都沒學

- 前五天（31、24、20、13、14 度）的答案是熱、冷、熱、冷、冷。24 度那天是「冷」，模型只給 0.332，損失 1.10；20 度那天是「熱」，模型給「冷」0.622，損失 0.97。照定義、穩定的寫法、EP07 的 `cross_entropy_loss`（第 0 類是「冷」，所以答案寫成 $1 - y$）三種寫法算出來都一樣。
- **什麼都沒學**（$w = b = 0$）：每一天 $p = 0.5$，損失 $\ln 2 = 0.693$，就像 EP07 的「151,936 個 token 一樣可能，損失 11.93」，只是這裡只有 2 類。

### 算得穩：不要先算機率再取 log

第一行是照定義的 sigmoid 在 $z = \pm 1000$；第二行是一天「熱」、模型卻很有把握說「冷」（$z = 1000$）的損失，左邊先算機率再取 log，右邊用穩定的寫法；第三行是穩定的 sigmoid（這格會跳出兩個 `RuntimeWarning`：那只是警告，不會讓 notebook 停下來）：

In [ ]:
z_big = np.array([-1000.0, 1000.0])
p_big = sigmoid_naive(z_big)                  # e^1000 溢位：會跳出 RuntimeWarning
print(p_big)
print(-np.log(1 - p_big[1]), np.logaddexp(0, 1000.0))   # 熱的一天，模型卻很有把握說冷（z = 1000）

def sigmoid(z):
    return np.exp(-np.logaddexp(0, -z))       # 1 / (1 + e^(−z)) = e^(−ln(1 + e^(−z)))：不會溢位

print(sigmoid(z_big), sigmoid(np.array([-2.0, 0.0, 2.0])))

- `sigmoid_naive(-1000)` 要算 $e^{1000}$，溢位（跳出 `RuntimeWarning: overflow encountered in exp`），$1 / (1 + \infty) = 0$ 碰巧是對的。
- 損失就糟了：一天「熱」、模型卻很有把握說「冷」（$z = 1000$），$p$ 存成剛好 1，$-\ln(1 - 1) = \infty$（跳出 `divide by zero encountered in log`）。正確答案是 $\ln(1 + e^{1000}) = 1000$，很大但有限；一天 `inf`，平均就是 `inf`。
- 穩定的 sigmoid 用 $\sigma(z) = e^{-\ln(1 + e^{-z})}$（因為 $\frac{1}{u} = e^{-\ln u}$），$e^{-1000}$ 只會下溢成 0，不跳警告，之後都用它。原則同 EP01、EP02：**直接從分數算，不要先算出機率再取 log**。

### 梯度：p − y

一天的損失 $\ell = \ln(1 + e^z) - yz$ 對 $z$ 微分。前項用鏈鎖律（EP05）：外層 $\ln u$ 的導數是 $\frac{1}{u}$，內層 $1 + e^z$ 的導數是 $e^z$（EP04）：

$$
\frac{d\ell}{dz} = \frac{e^z}{1 + e^z} - y = \sigma(z) - y = p - y
$$

白話：**就是 EP07 的 $q - y$，只看「冷」那一格**。

再接一層 $z = wx + b$：$\frac{\partial z}{\partial w} = x$、$\frac{\partial z}{\partial b} = 1$，所以一天的梯度是 $\bigl((p - y)\,x,\; p - y\bigr)$。$N$ 天平均：

$$
\frac{\partial\,\text{loss}}{\partial w} = \frac{1}{N} \sum_{i=1}^{N} (p_i - y_i)\,x_i, \qquad \frac{\partial\,\text{loss}}{\partial b} = \frac{1}{N} \sum_{i=1}^{N} (p_i - y_i)
$$

白話：每一天「猜的機率 − 答案」，乘上那天的特徵，再平均。

有 $d$ 個特徵時，把 $N$ 天的特徵排成 $(N, d)$ 的矩陣 $X$，$w$ 是 $d$ 個數，$z = Xw + b$，$w$ 的梯度是 $X^\top (p - y) / N$（EP03：$X^\top$ 是 $X$ 的轉置，等於讓 $X$ 的每一欄和 $p - y$ 做內積）。用 EP04 的數值梯度檢查。第一行左邊是手推的梯度、右邊是數值梯度（在 $(w, b) = (-3, 0.5)$）；第二行是從 $w = b = 0$ 出發的梯度：

In [ ]:
def loss(theta):                              # θ = (w, b)
    z = theta[0] * x + theta[1]
    return np.mean(np.logaddexp(0, z) - y * z)

def grad(theta):                              # 手推的梯度 (∂loss/∂w, ∂loss/∂b)
    p = sigmoid(theta[0] * x + theta[1])
    return np.array([np.mean((p - y) * x), np.mean(p - y)])

theta = np.array([-3.0, 0.5])
print(grad(theta), numerical_gradient(loss, theta))
print(grad(np.zeros(2)))

- 手推的和中央差分一樣（印出來的位數全部相同），這就是 EP04 說的梯度檢查（EP11 會拿它檢查自動微分）。
- 從 $w = b = 0$ 出發，每一天 $p = 0.5$。$b$ 的梯度 $0.5 - \frac{21}{40} = -0.025$：「冷」稍微多一點，負梯度把 $b$ 往上推。$w$ 的梯度 0.38 是正的：「冷」的日子 $x$ 大多是負的，$(0.5 - 1) \times$ 負數 $> 0$；「熱」的日子 $x$ 大多是正的，$(0.5 - 0) \times$ 正數也 $> 0$。負梯度把 $w$ 往負的推：**越冷越像「冷」**。

## 4. 工具四：損失地形

### 損失是參數的函數

資料固定，損失就只是 $(w, b)$ 的函數，就像 EP06 的概似函數 $L(p)$ 是參數 $p$ 的函數。兩個參數，可以像 EP04 的碗一樣畫成等高線圖，叫**損失地形 loss landscape**。用 EP01 的廣播一次算一整片，依序印出損失地形的形狀、最低的損失和它在哪一組 $(w, b)$，以及最高的損失：

In [ ]:
W, B = np.meshgrid(np.linspace(-10, 2, 121), np.linspace(-4, 4, 81))   # 每隔 0.1 一個點
Z = W[..., None] * x + B[..., None]           # (81, 121, 40)：每一組 (w, b) 對 40 天的分數
L = np.mean(np.logaddexp(0, Z) - y * Z, axis=-1)   # (81, 121)：損失地形
i = np.unravel_index(L.argmin(), L.shape)
print(L.shape, L.min(), W[i], B[i], L.max())

`W[..., None]` 在最後加一個軸（EP01），和 40 個 $x$ 廣播成 `(81, 121, 40)`；`np.unravel_index` 把 `argmin` 找到的位置換回列、欄。最低的格點是 $(w, b) \approx (-6.7, 1.0)$，損失 0.1845。畫出來是斜放的橢圓山谷（等一下的圖）：往右（$w > 0$，越熱越像「冷」）很快變高；往左（S 形太陡，對 20 到 24 度那幾天太有把握）慢慢變高。

### 凸：只有一個谷

從三個很不一樣的地方出發，各走 3000 步（學習率 1）。每一行是起點、終點和損失：

In [ ]:
for start in [(0.0, 0.0), (-10.0, 4.0), (2.0, -4.0)]:
    theta = np.array(start)
    for _ in range(3000):
        theta = theta - 1.0 * grad(theta)
    print(start, np.round(theta, 3), round(loss(theta), 8))

三個起點都走到 $(-6.71, 1.01)$，損失 0.18449：**只有一個谷**，從哪裡出發都走到同一個底。這種地形叫**凸 convex**：像一個碗，碗壁只往上彎，中間不會鼓起一座山，也就沒有別的山谷可以把你困住。邏輯迴歸的損失不管資料長怎樣都是凸的（證明這裡不講）。不過凸只保證「沒有別的谷」：不保證「有底」（資料完全分得開時，谷底在無限遠，弄壞它 5），也不保證一定走得到（學習率太大，會在谷底附近來回跳，弄壞它 1）。

畫出損失地形（等高線上的數字是損失）和三條路徑（用工具二的 `run`；前 30 步每一步都畫，之後挑 60 步）：

In [ ]:
WL, BL = np.meshgrid(np.linspace(-10.5, 2.5, 261), np.linspace(-4.5, 4.5, 181))   # 比上面的格子細一倍
ZL = WL[..., None] * x + BL[..., None]
LL = np.mean(np.logaddexp(0, ZL) - y * ZL, axis=-1)
fig, ax = plt.subplots(figsize=(8, 5))
cs = ax.contour(WL, BL, LL, levels=[0.19, 0.2, 0.25, 0.3, 0.4, 0.6, 0.8, 1.2, 2], colors="0.7", linewidths=0.9)
ax.clabel(cs, fmt="%g", fontsize=8, colors=MUTED)
for start, color in zip([(0.0, 0.0), (-10.0, 4.0), (2.0, -4.0)], SERIES):
    hist = run(grad, start, 1.0, 3000)                                        # 學習率 1，3000 步
    shown = hist[np.unique(np.r_[np.arange(0, 30), np.geomspace(30, 3000, 60).astype(int)])]
    ax.plot(shown[:, 0], shown[:, 1], ".-", color=color, markersize=4, linewidth=1.5,
            label=f"start ({start[0]:g}, {start[1]:g})")
    ax.plot([start[0]], [start[1]], "o", color=color, markersize=8)          # 起點
ax.plot([hist[-1, 0]], [hist[-1, 1]], "*", color=INK, markersize=14, zorder=5)   # 谷底
ax.annotate("bottom (-6.71, 1.01)\nloss 0.1845", hist[-1], xytext=(-40, -40), textcoords="offset points",
            color=INK, fontsize=9)
ax.set(xlim=(-10.5, 2.5), ylim=(-4.5, 4.5), xlabel="w", ylabel="b",
       title="loss landscape of logistic regression (demo data, standardized)")
tidy(ax, grid=False)
ax.legend(loc="lower left", frameon=True, framealpha=0.9, title="gradient descent, lr 1, 3000 steps",
          title_fontsize=9)
plt.show()

斜放的橢圓山谷；三個起點都走到谷底 $(-6.71, 1.01)$，損失 0.1845。

（網站上有互動元件：在這張損失地形上點任何地方當起點，學習率 1 的路徑都走到谷底；也可以切換成「不標準化」（直接用氣溫），看又長又窄、斜著放的谷，學習率 0.01 要快轉到 10 萬步才到分界 21.38 度。）

### LLM 的損失地形

邏輯迴歸只有 2 個參數、一個谷。LLM 的損失地形有約 5 億維（Qwen2.5-0.5B 約 5 億個參數），畫不出來，也不是凸的：有很多起伏，從哪裡出發會影響走到哪裡（EP13 的初始化 initialization）。梯度下降照樣有用，只是不保證走到最低點。

## 5. 本集重點：第一個會學習的模型

### 四步接成一個迴圈

$w$、$b$ 從 0 開始（什麼都沒學），學習率固定 1，一輪一輪做這四步：**① 算預測 → ② 算損失 → ③ 算梯度 → ④ 往負梯度走一步**。每一列印出：步數、損失、$w$、$b$、**準確率 accuracy**（猜對的比例）、分界氣溫：

In [ ]:
w, b, lr = 0.0, 0.0, 1.0                      # 從 0 開始；學習率固定 1
history = []                                  # 實驗紀錄：每一步的 (w, b, 損失)
for step in range(301):
    z = w * x + b
    p = sigmoid(z)                            # ① 算預測：每一天「冷」的機率
    loss_now = np.mean(np.logaddexp(0, z) - y * z)   # ② 算損失：平均二元交叉熵
    dw, db = np.mean((p - y) * x), np.mean(p - y)    # ③ 算梯度
    history.append((w, b, loss_now))
    if step in [0, 1, 10, 30, 100, 300]:
        acc = np.mean((p >= 0.5) == y)        # 準確率：p ≥ 0.5 判成「冷」
        cut = 20 - 10 * b / w if w != 0 else np.nan   # 分界氣溫：p = 0.5 的地方
        print(step, round(loss_now, 4), round(w, 3), round(b, 3), acc, round(cut, 2))
    w, b = w - lr * dw, b - lr * db           # ④ 往負梯度走一步
history = np.array(history)                   # (301, 3)

這個迴圈叫**訓練迴圈 training loop**；每一步都用全部 40 天算梯度，叫**全批次 full-batch**。

- **損失每一步都下降**：0.693（什麼都沒學）→ 0.564 → 0.296 → 0.196 → 0.186，越來越靠近谷底的 0.1845。
- 第 0 步 $p$ 全是 0.5，全部判成「冷」，準確率 $\frac{21}{40} = 0.525$。**第 1 步準確率就到 0.9**，分界 20.66 度，之後 299 步準確率都沒變。
- 那後面 299 步在做什麼？$w$ 從 −0.38 變成 −5.75：**S 形越來越陡，模型越來越有把握**，分界從 20.66 度慢慢移到 21.46 度。準確率只看 $p$ 落在 0.5 的哪一邊，損失看正確答案拿到多少機率。訓練要用損失：準確率一格一格跳，參數動一點點它通常不變，幾乎每個地方的梯度都是 0，指不出方向。

### S 形曲線貼上資料

前四行是第 0、1、10、300 步時，5、15、20、25、35 度「冷」的機率；最後一行是第 300 步猜錯的日子：氣溫、答案、模型給「冷」的機率：

In [ ]:
temps = np.array([5, 15, 20, 25, 35])
for step in [0, 1, 10, 300]:
    w_, b_ = history[step, :2]
    print(step, np.round(sigmoid(w_ * (temps - 20) / 10 + b_), 3))   # 這幾個氣溫「冷」的機率
w_, b_ = history[300, :2]
p300 = sigmoid(w_ * x + b_)
wrong = (p300 >= 0.5) != y                    # 猜錯的日子
print(temp[wrong], y[wrong], np.round(p300[wrong], 3))

- 第 0 步是 0.5 的水平線，第 1 步微微傾斜，第 10 步是平緩的 S，第 300 步是陡的 S。
- 第 300 步猜錯的 4 天都在 20 到 24 度：24 度「冷」、20 度「熱」、21 度「熱」、22 度「冷」。同樣的氣溫有冷有熱，**沒有任何一條 S 形能全部猜對**，損失也降不到 0：這是資料本身的不確定性（EP07：損失的地板是熵）。
- 資料是照「20 度」的規則產生的，學到的分界卻是 21.5 度：只有 40 天，這 40 天剛好偏了一點（網站上的自我檢測 6 會用 400 天、4000 天再試）。

本集重點那張圖（左：學習率 0.1、1、10 各走 300 步的損失；右：40 天的點，和學習率 1 第 0、10、300 步的 S 形曲線）在第 6 節動手做畫：網站上是 `plot_training.py`，這裡用你寫的 `draw_loss`、`draw_fit`，先自己寫寫看（參考解答在最後）。

（網站上有互動元件：在示範資料 40 天的點上，按「走一步」「走 10 步」「重來」自己一步一步訓練，看 S 形曲線、分界氣溫、步數、損失、準確率和損失的小圖；學習率可選 0.01、1、10、100，100 會來回跳。）

### 和 LLM 的連結

沒有人告訴模型「21.5 度」或「越冷越像冷」，只給了資料、損失和「往負梯度走」，$w$、$b$ 就從 0 走到能解釋資料的地方：這就是**從資料學習**，也是 EP00 的「猜錯了就調整參數」（猜得越錯，$p - y$ 離 0 越遠，改得越多）。

- **這個迴圈就是訓練 LLM 的骨架**：① 從前文算出 151,936 個分數、做 Softmax → ② 算平均交叉熵（EP07）→ ③ 算梯度 → ④ 往負梯度走一步，做非常多輪。
- **LLM 的最後一步就是多類別的邏輯迴歸**（EP00 的 ⑤ 打分數＋⑥ 換成機率）：特徵是最後一個向量的 896 個數，分數是 EP03 的 $(151936, 896)$ 矩陣乘上它。兩類換成 151,936 類、sigmoid 換回 Softmax、$p - y$ 換回 $q - y$，就是它，也叫 **Softmax 迴歸 softmax regression**（EP12 會用它分手寫數字的 10 類）。
- **LLM 和這集哪裡不一樣**：
  1. **約 5 億個參數、很多層**：特徵是前面的層算出來的，也要學；手推梯度不可能，要靠反向傳播（EP09–EP11）。
  2. **一次只看一小批資料**：每一步只用一小批資料算梯度，叫 **mini-batch**（EP12）。
  3. **更聰明的走法**：動量 momentum、Adam、學習率排程（EP15）；地形不是凸的，初始化（EP13）、過擬合 overfitting 與正則化 regularization（EP16）也會影響結果。

## 6. 動手做：從零寫出 `gd.py` 的 7 個函式

在下一格寫完這 7 個函式（網站上是寫在 `llm-labs/ep08/gd.py`），再執行下一格的檢查。

要求與提示：

1. **只用 NumPy、基本的 Python 和前幾集自己寫的函式**。
2. **`make_weather_data`**：照抄工具三示範資料那一格的前四行，把 `default_rng(0)` 的 0 換成 `seed`、40 換成 `n`。**亂數的呼叫順序不能換**（先 `integers`、再 `uniform`），檢查會看前五天、「冷」剛好 21 天，以及換了 seed 資料會不一樣。
3. **`gradient_descent`**：先 `theta = np.array(theta0, dtype=float)` 複製一份，每一步 `theta = theta - lr * grad_fn(theta)` 產生新陣列，把 `theta.copy()` 加進 `history`（list），最後 `np.array(history)`。工具二的 `run` 就是它。
4. **`sigmoid`**：用工具三「算得穩」那一段的寫法；或照正負號分開算：`e = np.exp(-np.abs(z))`，$z \ge 0$ 用 `1 / (1 + e)`、$z < 0$ 用 `e / (1 + e)`（上下同乘 $e^z$），再用 `np.where` 挑。不要在 `np.where` 裡直接寫 `np.exp(-z)` 和 `np.exp(z)`：兩邊都會先算完，一樣溢位。
5. **`bce_loss`**：`float(np.mean(np.logaddexp(0, z) - y * z))`，`z = X @ w + b`；`X`、`y`、`w` 先 `np.asarray(..., dtype=float)`（檢查會傳 list）。
6. **`bce_grad`**：`r = sigmoid(X @ w + b) - y`，`dw = X.T @ r / N`、`db = float(np.mean(r))`。記得除以 $N$（弄壞它 3）。
7. **`fit_logistic`**：把 $w$、$b$ 排成一排 `theta`（長度 $d + 1$，最後一個是 $b$），`grad_fn(theta)` 回傳 `np.append(dw, db)`（把 `db` 接在 `dw` 後面），交給 `gradient_descent`；再對 `history` 的每一列算 `bce_loss`。
8. **`accuracy`**：`sigmoid(...) >= 0.5` 和 `y == 1` 比，取平均。

注意：這一格的 `sigmoid` 會取代工具三「算得穩」那一格定義的 `sigmoid`（網站上它們在不同的檔案，不會互相影響）。還沒寫好就回頭重新執行前面用到 `sigmoid` 的格子（工具三的梯度檢查、工具四、本集重點），會出現「還沒寫好 sigmoid」：先寫好它，或重新執行「算得穩」那一格。

In [ ]:
def make_weather_data(n=40, seed=0):
    """示範資料（工具三的規則）：n 天的氣溫（°C，5 到 35 的整數）和答案 y（1 = 冷、0 = 熱）。回傳 (temp, y)。"""
    # 你的實作
    raise NotImplementedError("還沒寫好 make_weather_data")


def gradient_descent(grad_fn, theta0, lr, steps):
    """每一步 θ ← θ − lr × grad_fn(θ)，走 steps 步。回傳 (theta, history)：最後的參數，和每一步的紀錄
    （theta0 是 d 個數時形狀 (steps + 1, d)；history[k] 是走了 k 步之後，history[0] 是起點）。不能改到呼叫端的 theta0。"""
    # 你的實作
    raise NotImplementedError("還沒寫好 gradient_descent")


def sigmoid(z):
    """σ(z) = 1 / (1 + e^(−z))，z 可以是一個數或陣列。數值穩定：z = ±1000 不能有警告、inf 或 nan。"""
    # 你的實作
    raise NotImplementedError("還沒寫好 sigmoid")


def bce_loss(w, b, X, y):
    """平均二元交叉熵（nat，float）。X：(N, d)；w：(d,)；b：一個數；y：長度 N 的 0／1。用穩定的寫法。"""
    # 你的實作
    raise NotImplementedError("還沒寫好 bce_loss")


def bce_grad(w, b, X, y):
    """bce_loss 對 w、b 的梯度 (dw, db) = (Xᵀ(p − y) / N, mean(p − y))。dw 形狀 (d,)，db 是 float。"""
    # 你的實作
    raise NotImplementedError("還沒寫好 bce_grad")


def fit_logistic(X, y, lr, steps):
    """w、b 從 0 開始，用 gradient_descent 走 steps 步。回傳 (w, b, losses)：w 形狀 (d,)、b 是 float，
    losses 長度 steps + 1（losses[k] 是走了 k 步之後的損失）。"""
    # 你的實作
    raise NotImplementedError("還沒寫好 fit_logistic")


def accuracy(w, b, X, y):
    """p ≥ 0.5 判成 1、否則判成 0，回傳猜對的比例（float）。"""
    # 你的實作
    raise NotImplementedError("還沒寫好 accuracy")

下一格的 7 個檢查和網站上的 `tests/test_ep08_gd.py` 一樣，依序檢查：

1. `gradient_descent` 在碗 $x^2 + 3y^2$ 上：學習率 0.1 的前三步是 $(1, 2)$、$(0.8, 0.8)$、$(0.64, 0.32)$，100 步後收斂，不能改到呼叫端的起點；0.3 每步換邊、但會收斂；0.34 越跳越遠。
2. `sigmoid`：$z = \pm 1000$ 不能有警告、`inf` 或 `nan`；$\sigma(z) + \sigma(-z) = 1$；等於分數 $[z, 0]$ 的 Softmax 取第一格；一個數也要能算。
3. `bce_loss`：什麼都沒學是 $\ln 2$；一筆手算的例子；要回傳 `float`；和 EP07 的 `cross_entropy_loss` 一樣；很有把握地答錯是 1000（大但有限）、答對是 0，都沒有警告。
4. `bce_grad`：形狀對，和中央差分的梯度一樣（梯度檢查）。
5. `fit_logistic`：示範資料、學習率 1 走 300 步，損失從 $\ln 2$ 出發、每一步都下降、停在 0.18 到 0.19；準確率 0.9；分界氣溫在 21 到 22 度；不能改到呼叫端；學習率 0.1 只到 0.2308（學習率要真的用上）。
6. `make_weather_data`：固定 seed 可重現、21 天「冷」、前五天和工具三一樣、`seed` 和 `n` 要真的用上。
7. 和 PyTorch 比對（沒裝 PyTorch 就跳過）：PyTorch 跑同樣的 300 步，`w`、`b` 要一樣到 $10^{-10}$。

幾個寫法：

- `warnings.simplefilter("error")` 把警告變成錯誤；放在 `with warnings.catch_warnings():` 底下，只影響那一段。所以「不能有警告」也檢查得到。
- `type(...) is float` 檢查回傳的是 Python 的 `float`，不是 `np.float64`。
- 和 PyTorch 比對時兩邊只差捨入誤差，所以用很嚴的 `rel_tol=1e-12`、`1e-10`。

沒通過的檢查會印出出錯的那一行。改了上一格之後，要重新執行上一格，再執行這一格。

In [ ]:
import traceback
import warnings


def check_gd():
    """和網站上 tests/test_ep08_gd.py 一樣的 7 個測試。"""

    class Skipped(Exception):
        pass

    def bowl_grad(p):
        return np.array([2 * p[0], 6 * p[1]])                     # EP04 的碗 x² + 3y² 的梯度

    def weather():
        temp, y = make_weather_data()                             # 示範資料：40 天，seed 0
        return ((temp - 20) / 10).reshape(-1, 1), y               # 標準化，形狀 (40, 1)

    def gradient_descent_on_bowl():
        start = np.array([1.0, 2.0])
        theta, hist = gradient_descent(bowl_grad, start, lr=0.1, steps=100)
        assert hist.shape == (101, 2) and np.array_equal(theta, hist[-1])
        assert np.allclose(hist[:3], [[1, 2], [0.8, 0.8], [0.64, 0.32]])   # x ← 0.8x、y ← 0.4y
        assert np.linalg.norm(theta) < 1e-8 and np.array_equal(start, [1.0, 2.0])   # 收斂；不能改到呼叫端
        _, hist = gradient_descent(bowl_grad, start, lr=0.3, steps=50)
        assert np.all(hist[1:, 1] * hist[:-1, 1] < 0) and abs(hist[-1, 1]) < 1e-4   # 每步換邊，但會收斂
        _, hist = gradient_descent(bowl_grad, start, lr=0.34, steps=50)
        assert np.all(np.abs(hist[1:, 1]) > np.abs(hist[:-1, 1]))   # 1 − 6η = −1.04：越跳越遠

    def sigmoid_is_stable_and_two_class_softmax():
        with warnings.catch_warnings():
            warnings.simplefilter("error")                        # 有警告就算失敗
            out = sigmoid(np.array([-1000.0, 0.0, 1000.0]))
        assert np.all(np.isfinite(out)) and np.allclose(out, [0.0, 0.5, 1.0])
        z = np.random.default_rng(0).normal(size=20) * 5
        assert np.allclose(sigmoid(z) + sigmoid(-z), 1.0)         # 冷 + 熱 = 1
        assert np.allclose(sigmoid(z), softmax(np.stack([z, 0 * z], axis=1))[:, 0])   # 分數 [z, 0] 的 Softmax
        assert math.isclose(sigmoid(2.0), 1 / (1 + math.exp(-2)))   # 一個數也要能算

    def bce_loss_hand_computed():
        X, y = weather()
        assert math.isclose(bce_loss(np.zeros(1), 0.0, X, y), math.log(2))   # 什麼都沒學：ln 2
        assert math.isclose(bce_loss([2.0], -1.0, [[1.0]], [1]), math.log(1 + math.exp(-1)))   # z = 1、冷
        assert type(bce_loss([2.0], -1.0, [[1.0]], [1])) is float   # 要回傳 float，不是 np.float64
        z = -3.0 * X[:, 0] + 0.5                                  # 和 EP07 的交叉熵一樣：分數 [z, 0]
        assert math.isclose(bce_loss([-3.0], 0.5, X, y), cross_entropy_loss(np.stack([z, 0 * z], axis=1), 1 - y))
        with warnings.catch_warnings():
            warnings.simplefilter("error")
            assert bce_loss([1.0], 0.0, [[1000.0]], [0]) == 1000.0     # 很有把握地答錯：大但有限
            assert bce_loss([1.0], 0.0, [[1000.0]], [1]) == 0.0

    def bce_grad_matches_numerical_gradient():
        rng = np.random.default_rng(1)
        X, y = rng.normal(size=(8, 3)), rng.integers(0, 2, size=8)
        w, b = rng.normal(size=3), 0.3
        dw, db = bce_grad(w, b, X, y)
        assert dw.shape == (3,) and isinstance(db, float)
        assert np.allclose(dw, numerical_gradient(lambda v: bce_loss(v, b, X, y), w), rtol=0, atol=1e-8)
        assert math.isclose(db, numerical_gradient(lambda c: bce_loss(w, c[0], X, y), np.array([b]))[0], abs_tol=1e-8)

    def fit_logistic_learns_weather():
        X, y = weather()
        X0, y0 = X.copy(), y.copy()
        w, b, losses = fit_logistic(X, y, lr=1.0, steps=300)
        assert w.shape == (1,) and isinstance(b, float) and losses.shape == (301,)
        assert math.isclose(losses[0], math.log(2)) and np.all(np.diff(losses) <= 0)   # 從 0 開始，每步都下降
        assert math.isclose(losses[-1], bce_loss(w, b, X, y)) and 0.18 < losses[-1] < 0.19
        assert accuracy(w, b, X, y) == 0.9
        assert accuracy(np.zeros(1), 0.0, X, y) == y.mean()       # p = 0.5：全部判成「冷」
        assert 21 < 20 - 10 * b / w[0] < 22                       # 分界氣溫：p = 0.5 的地方
        assert np.array_equal(X, X0) and np.array_equal(y, y0)    # 不能改到呼叫端
        assert 0.23 < fit_logistic(X, y, lr=0.1, steps=300)[2][-1] < 0.232   # lr 要真的用上：0.1 走 300 步只到 0.2308

    def reproducible_with_fixed_seed():
        (t1, y1), (t2, y2) = make_weather_data(), make_weather_data()
        assert np.array_equal(t1, t2) and np.array_equal(y1, y2)
        assert t1.shape == (40,) and t1.min() >= 5 and t1.max() <= 35 and y1.sum() == 21   # 21 天「冷」
        assert not np.array_equal(make_weather_data(seed=1)[0], t1)   # seed 要真的用上
        assert t1[:5].tolist() == [31, 24, 20, 13, 14] and y1[:5].tolist() == [0, 1, 0, 1, 1]   # 工具三印出來的前五天
        t400, y400 = make_weather_data(n=400)
        assert t400.shape == y400.shape == (400,)                 # n 要真的用上

    def matches_pytorch():
        try:
            import torch
        except ImportError:
            raise Skipped("沒有安裝 PyTorch")
        F = torch.nn.functional
        X, y = weather()
        Xt, yt = torch.tensor(X), torch.tensor(y, dtype=torch.float64)
        w = torch.zeros(1, dtype=torch.float64, requires_grad=True)
        b = torch.zeros(1, dtype=torch.float64, requires_grad=True)
        opt = torch.optim.SGD([w, b], lr=1.0)                     # 每一步用全部 40 筆：全批次梯度下降
        for _ in range(300):
            opt.zero_grad()
            loss = F.binary_cross_entropy_with_logits(Xt @ w + b, yt)
            loss.backward()
            opt.step()
        w_np, b_np, losses = fit_logistic(X, y, lr=1.0, steps=300)
        assert math.isclose(loss.item(), losses[-2], rel_tol=1e-12)   # 最後一次 backward 是第 299 步
        assert np.allclose(w.detach().numpy(), w_np, rtol=1e-10) and math.isclose(b.item(), b_np, rel_tol=1e-10)

    tests = [gradient_descent_on_bowl, sigmoid_is_stable_and_two_class_softmax, bce_loss_hand_computed,
             bce_grad_matches_numerical_gradient, fit_logistic_learns_weather, reproducible_with_fixed_seed,
             matches_pytorch]
    passed = skipped = pending = 0
    for t in tests:
        try:
            t()
            print("✓", t.__name__)
            passed += 1
        except NotImplementedError as e:
            print("…", t.__name__, f"（{e}）")
            pending += 1
        except Skipped as e:
            print("-", t.__name__, f"跳過（{e}）")
            skipped += 1
        except Exception as e:
            msg = f"{type(e).__name__}: {e}" if str(e) else type(e).__name__
            frames = [fr for fr in traceback.extract_tb(e.__traceback__) if fr.name == t.__name__]
            line = frames[-1].line if frames else ""                    # 檢查裡出錯的那一行
            print("✗", t.__name__, msg, f"｜{line}" if line else "")
    summary = f"{passed}/{len(tests)} 通過"
    if skipped:
        summary += f"，{skipped} 個跳過"
    if pending:
        summary += f"，{pending} 個還沒寫好：先完成上一格，或執行最後的參考解答"
    print(summary)


check_gd()

### 畫出訓練過程：`weather_data`、`train`、`draw_loss`、`draw_fit`

網站上這是一個畫圖腳本 `llm-labs/ep08/plot_training.py`，執行後存成 `ep08/training.png`，也就是本集重點那張圖；在 notebook 裡只要在下一格寫好這 4 個函式，再下一格會用它們畫一列兩張圖（左邊 `draw_loss`、右邊 `draw_fit`），直接顯示。`STEPS` 是步數，`RATES` 是左圖的三個學習率，`SHOW` 是右圖要畫的步數。

要求與提示：

1. **示範資料、sigmoid 和訓練迴圈都在這一格自己寫**，不呼叫上面的 `make_weather_data`、`sigmoid`、`fit_logistic`：網站上的腳本是直接執行的，Python 只在 `ep08/` 裡找模組，找不到 `ep08.gd`（同 EP06、EP07）。自己寫的小函式別取名叫 `sigmoid`，會蓋掉上一格的 `sigmoid`（例如叫 `sigmoid_plot`）。`train` 照本集重點的訓練迴圈寫，每一列是走了 $k$ 步之後的 $(w, b, \text{損失})$。
2. **左圖**：每個學習率 `ax.plot(損失)` 畫一條線；`ax.annotate(f"{最後的損失:.4f}", (STEPS, 最後的損失), xytext=(4, dy), textcoords="offset points")` 在右端寫出數字，1 和 10 最後的損失很接近（只差 0.0016），用不同的 `dy` 把兩個數字上下挪開一點；`ax.axhline(np.log(2), linestyle=":")` 畫 ln 2 的虛線。$y$ 軸標籤寫上單位（nat）。
3. **右圖**：`ax.scatter(temp, y, alpha=0.35)` 畫 40 天的點；S 形曲線用很多個氣溫（例如 3 到 37 度取 341 個點）換成標準化的 $x$ 再算機率，顏色由淺到深；第 `STEPS` 步的分界氣溫 $20 - \frac{10b}{w}$ 用 `ax.axvline(..., linestyle="--")` 畫垂直虛線，再用 `ax.annotate` 標出數字（例如 `f"{cut:.1f}"`）。

In [ ]:
STEPS = 300
RATES = [0.1, 1.0, 10.0]                            # 左圖的三個學習率
SHOW = [0, 10, 300]                                 # 右圖要畫的步數（學習率 1）


def weather_data(n=40, seed=0):
    """示範資料，和 gd.py 的 make_weather_data 一樣。回傳 (temp, y)。"""
    # 你的實作
    raise NotImplementedError("還沒寫好 weather_data")


def train(x, y, lr, steps):
    """w、b 從 0 開始做全批次梯度下降（x 是標準化後的氣溫）；回傳每一步的 (w, b, 損失)，形狀 (steps + 1, 3)。"""
    # 你的實作
    raise NotImplementedError("還沒寫好 train")


def draw_loss(ax, x, y):
    """左圖：RATES 的每個學習率各走 STEPS 步的損失曲線（nat），右端寫出最後的損失；一條 ln 2 的虛線（什麼都沒學）。"""
    # 你的實作
    raise NotImplementedError("還沒寫好 draw_loss")


def draw_fit(ax, temp, x, y):
    """右圖：40 天的點（縱軸 1 = 冷、0 = 熱，半透明），學習率 1 在 SHOW 那幾步的 S 形曲線（由淺到深），
    第 STEPS 步的分界氣溫畫垂直虛線、標出數字。"""
    # 你的實作
    raise NotImplementedError("還沒寫好 draw_fit")

下一格用你的 4 個函式畫圖，再檢查畫出來的東西。畫對的話就是本集重點那張圖：左圖三條線從 0.693 出發，停在 0.2308、0.1861、0.1845；右圖第 300 步的 S 形在 21.5 度穿過 0.5。最後一個檢查換一組設定（`RATES = [0.5, 3.0]`、`STEPS = 50`、`SHOW = [0, 50]`）再畫一次（不顯示），確認數字是算出來的，不是寫死的；畫完會把 `RATES`、`STEPS`、`SHOW` 改回來。

In [ ]:
import re

import matplotlib.colors as mcolors


def show_training_plot():
    """一列兩張圖：左邊 draw_loss、右邊 draw_fit，再檢查示範資料、訓練迴圈、曲線、點、文字和座標軸。"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.4))
    try:
        temp_d, y_d = weather_data()
        x_d = (temp_d - 20) / 10                             # 標準化
        draw_loss(ax1, x_d, y_d)
        draw_fit(ax2, temp_d, x_d, y_d)
    except NotImplementedError as e:
        plt.close(fig)                  # 不顯示空白的圖
        print(f"{e}：先完成上一格，或執行最後的參考解答，再回來執行這格")
        return

    rng = np.random.default_rng(0)                           # 檢查自己產生一份示範資料（同工具三）
    temp_ref = rng.integers(5, 36, size=40)
    y_ref = (temp_ref + rng.uniform(-6, 6, size=40) < 20).astype(int)
    x_ref = (temp_ref - 20) / 10

    def ref_train(lr, steps):                                # 檢查自己算一份：每一步的 (w, b, 損失)
        w, b, rows = 0.0, 0.0, []
        for _ in range(steps + 1):
            z = w * x_ref + b
            rows.append((w, b, np.mean(np.logaddexp(0, z) - y_ref * z)))
            p = np.exp(-np.logaddexp(0, -z))
            w, b = w - lr * np.mean((p - y_ref) * x_ref), b - lr * np.mean(p - y_ref)
        return np.array(rows)

    def prob_cold(w, b, t):                                  # 氣溫 t「冷」的機率
        return np.exp(-np.logaddexp(0, -(w * (t - 20) / 10 + b)))

    def cut_of(rows):                                        # 最後一步的分界氣溫
        return 20 - 10 * rows[-1, 1] / rows[-1, 0]

    def xy(line):
        return np.asarray(line.get_xdata(), dtype=float), np.asarray(line.get_ydata(), dtype=float)

    def loss_curve(ax, losses):                              # 有沒有一條線：x 是 0, 1, ⋯（步數），y 是損失
        for line in ax.get_lines():
            xs, ys = xy(line)
            if xs.shape == losses.shape and np.allclose(xs, np.arange(len(losses))) \
                    and np.allclose(ys, losses, rtol=1e-9, atol=1e-12):
                return True
        return False

    def dashed(line):
        return line.get_linestyle() not in ("-", "solid", "None", "", " ")

    def hline(ax, v):                                        # 水平虛線：y 都是 v
        return any(len(line.get_ydata()) >= 2 and np.allclose(xy(line)[1], v, atol=1e-9) and dashed(line)
                   for line in ax.get_lines())

    def vline(ax, v):                                        # 垂直虛線：x 都是 v
        return any(len(line.get_xdata()) >= 2 and np.allclose(xy(line)[0], v, atol=1e-6) and dashed(line)
                   for line in ax.get_lines())

    def s_curve(ax, w, b):                                   # 一條 S 形曲線：點數夠多、涵蓋 5 到 35 度、y 是「冷」的機率
        for line in ax.get_lines():
            xs, ys = xy(line)
            if len(xs) >= 50 and xs.min() <= 5 and xs.max() >= 35 \
                    and np.allclose(ys, prob_cold(w, b, xs), rtol=1e-7, atol=1e-9):
                return line
        return None

    def shade(line):                                         # 畫在白底上看起來多淺（1 是白、0 是黑）
        r, g, b_ = mcolors.to_rgb(line.get_color())
        alpha = 1.0 if line.get_alpha() is None else line.get_alpha()
        return 1 - alpha * (1 - (0.2126 * r + 0.7152 * g + 0.0722 * b_))

    def light_to_dark(ax, rows, steps):                      # 每一步都有 S 形曲線，而且由淺到深
        found = [s_curve(ax, rows[s, 0], rows[s, 1]) for s in steps]
        return all(found) and all(shade(a) > shade(b) for a, b in zip(found, found[1:]))

    def points(ax):                                          # 畫出來的點：scatter，或只有 marker 的線；附上透明度
        pts = []
        for c in ax.collections:
            alpha = c.get_alpha()
            fc = c.get_facecolor()
            a = alpha if alpha is not None else (float(fc[0][3]) if len(fc) else 1.0)
            pts += [(round(float(px), 6), round(float(py), 6), a) for px, py in c.get_offsets()]
        for line in ax.get_lines():
            if line.get_marker() not in (None, "None", "", " ") and line.get_linestyle() in ("None", "", " "):
                a = line.get_alpha() if line.get_alpha() is not None else mcolors.to_rgba(line.get_markerfacecolor())[3]
                pts += [(round(float(px), 6), round(float(py), 6), a) for px, py in zip(*xy(line))]
        return pts

    def data_points(ax):                                     # 40 天的點都畫了，而且半透明
        pts = points(ax)
        need = sorted((float(t), float(v)) for t, v in zip(temp_ref, y_ref))
        got = sorted((px, py) for px, py, _ in pts)
        it = iter(got)
        return all(any(g == n for g in it) for n in need) and all(a < 1 for _, _, a in pts)

    def texts(ax):
        return [t.get_text() for t in ax.texts]

    def says(ax, value):                                     # 有沒有一段文字寫著 value（4 位小數）
        return any(re.search(re.escape(f"{value:.4f}") + r"(?!\d)", s) for s in texts(ax))

    def apart(ax, a, b):                                     # 寫著 a 和寫著 b 的兩段文字沒有疊在一起
        ax.figure.canvas.draw()                              # 先畫一次，才知道每段文字在圖上的位置
        ta = [t for t in ax.texts if f"{a:.4f}" in t.get_text()]
        tb = [t for t in ax.texts if f"{b:.4f}" in t.get_text()]
        return bool(ta and tb) and not ta[0].get_window_extent().overlaps(tb[0].get_window_extent())

    rows = {lr: ref_train(lr, 300) for lr in [0.1, 1.0, 10.0]}
    finals = [rows[lr][-1, 2] for lr in [0.1, 1.0, 10.0]]
    cut = cut_of(rows[1.0])
    other = {}

    def draw_other():                                        # 換一組設定再畫一次，不顯示；畫完改回來
        global RATES, STEPS, SHOW
        if not other:
            saved = RATES, STEPS, SHOW
            RATES, STEPS, SHOW = [0.5, 3.0], 50, [0, 50]
            fig2, (b1, b2) = plt.subplots(1, 2, figsize=(11, 4.4))
            try:
                draw_loss(b1, x_d, y_d)
                draw_fit(b2, temp_d, x_d, y_d)
            finally:
                RATES, STEPS, SHOW = saved
                plt.close(fig2)
            other["axes"] = b1, b2
        return other["axes"]

    def other_ok():
        b1, b2 = draw_other()
        alt = {lr: ref_train(lr, 50) for lr in [0.5, 3.0]}          # 左圖：學習率 0.5、3 各走 50 步
        fit50 = ref_train(1.0, 50)                                   # 右圖：學習率 1 走 50 步
        return (all(loss_curve(b1, alt[lr][:, 2]) and says(b1, alt[lr][-1, 2]) for lr in alt)
                and light_to_dark(b2, fit50, [0, 50]) and vline(b2, cut_of(fit50)))

    def summary(ax):                                         # 沒通過時印出來：每條線的（點數, 第一個 y, 最後一個 y）
        return [(len(line.get_xdata()), round(float(xy(line)[1][0]), 4), round(float(xy(line)[1][-1]), 4))
                for line in ax.get_lines() if len(line.get_xdata())]

    checks = [
        ("weather_data：和工具三一樣的示範資料（前五天 31、24、20、13、14 度，21 天「冷」），n、seed 要真的用上",
         lambda: (np.array_equal(temp_d, temp_ref) and np.array_equal(y_d, y_ref)
                  and weather_data(n=400)[0].shape == (400,) and not np.array_equal(weather_data(seed=1)[0], temp_ref)),
         lambda: (np.asarray(temp_d)[:5].tolist(), np.asarray(y_d)[:5].tolist(), int(np.sum(y_d)))),
        ("train：形狀 (301, 3)，學習率 1 的第 0 步 (0, 0, 0.6931)、第 1 步 (−0.38, 0.025, 0.5638)、第 300 步損失 0.1861",
         lambda: np.allclose(np.asarray(train(x_ref, y_ref, 1.0, 300), dtype=float), rows[1.0], rtol=1e-9, atol=1e-12),
         lambda: np.round(np.asarray(train(x_ref, y_ref, 1.0, 300), dtype=float)[[0, 1, -1]], 4).tolist()),
        ("train：學習率和步數要真的用上（0.1 走 300 步停在 0.2308；走 50 步是 51 列）",
         lambda: (np.allclose(np.asarray(train(x_ref, y_ref, 0.1, 300), dtype=float), rows[0.1], rtol=1e-9, atol=1e-12)
                  and np.shape(train(x_ref, y_ref, 1.0, 50)) == (51, 3)),
         lambda: (np.round(np.asarray(train(x_ref, y_ref, 0.1, 300), dtype=float)[-1], 4).tolist(),
                  np.shape(train(x_ref, y_ref, 1.0, 50)))),
        ("左圖：學習率 0.1、1、10 各一條損失曲線（第 0 到 300 步，301 個點）",
         lambda: all(loss_curve(ax1, rows[lr][:, 2]) for lr in rows),
         lambda: "每條線的（點數, 第一個 y, 最後一個 y）" + str(summary(ax1))),
        ("左圖：右端寫出最後的損失 0.2308、0.1861、0.1845",
         lambda: all(says(ax1, v) for v in finals), lambda: texts(ax1)),
        ("左圖：1 和 10 右端的數字（0.1861、0.1845）上下挪開，沒有疊在一起",
         lambda: apart(ax1, finals[1], finals[2]), lambda: texts(ax1)),
        ("左圖：ln 2 = 0.693 的水平虛線（什麼都沒學）", lambda: hline(ax1, math.log(2)),
         lambda: "每條線的（點數, 第一個 y, 最後一個 y）" + str(summary(ax1))),
        ("左圖：y 軸標籤寫上單位 nat", lambda: "nat" in ax1.get_ylabel(), lambda: repr(ax1.get_ylabel())),
        ("右圖：40 天的點（氣溫, 1 = 冷、0 = 熱），半透明",
         lambda: data_points(ax2), lambda: f"{len(points(ax2))} 個點，透明度 {sorted({a for *_, a in points(ax2)})}"),
        ("右圖：學習率 1 第 0、10、300 步的 S 形曲線（涵蓋 5 到 35 度），由淺到深",
         lambda: light_to_dark(ax2, rows[1.0], [0, 10, 300]),
         lambda: "每條線的（點數, 第一個 y, 最後一個 y）" + str(summary(ax2))),
        (f"右圖：第 300 步的分界氣溫 {cut:.2f} 度畫垂直虛線，標出數字",
         lambda: vline(ax2, cut) and any(f"{cut:.1f}" in s or f"{cut:.2f}" in s for s in texts(ax2)),
         lambda: ("每條線的 x：" + str([np.round(xy(line)[0][:2], 3).tolist() for line in ax2.get_lines()
                                         if len(line.get_xdata()) <= 2]), texts(ax2))),
        ("右圖：x 軸包含 5 到 35 度，y 軸包含 0 到 1",
         lambda: (ax2.get_xlim()[0] <= 5 and ax2.get_xlim()[1] >= 35
                  and ax2.get_ylim()[0] <= 0 and ax2.get_ylim()[1] >= 1),
         lambda: (tuple(round(float(v), 2) for v in ax2.get_xlim()), tuple(round(float(v), 2) for v in ax2.get_ylim()))),
        ("換一組設定（RATES = [0.5, 3.0]、STEPS = 50、SHOW = [0, 50]）：曲線、右端的數字、S 形和分界跟著變",
         other_ok,
         lambda: (summary(draw_other()[0]), texts(draw_other()[0]), summary(draw_other()[1]))),
    ]
    results = []
    for name, ok, got in checks:                             # 先檢查完，再顯示圖
        try:
            passed = bool(ok())
        except Exception as e:                               # 例如畫圖時出錯，連比都沒辦法比
            passed, got = False, (lambda e=e: f"{type(e).__name__}: {e}")
        results.append((name, passed, got))
    plt.show()
    for name, passed, got in results:
        if passed:
            print("✓", name)
        else:
            try:
                print("✗", f"{name}：現在是 {got()}")
            except Exception as e:
                print("✗", f"{name}：{type(e).__name__}: {e}")


show_training_plot()

左圖也說明了學習率為什麼挑 1：0.1 走 300 步只降到 0.2308，10 不到 100 步就到谷底，挑 1 是為了看清楚「幾百步慢慢變好」。300 步停在 $(-5.75, 0.84)$，還沒到底，但損失只比最低的 0.1845 多 0.0016。

網站上的腳本把主程式放在 `if __name__ == "__main__":` 底下，用 `matplotlib.use("Agg")` 只存檔、不開視窗，並寫死 `figsize` 和 `dpi`，執行兩次的 `md5sum` 指紋一樣（同 EP04–EP07）；在 notebook 裡用 `plt.show()` 直接顯示就好。

### 和 PyTorch 對照

用你在上面寫的 `fit_logistic`，和 PyTorch 跑同樣的 300 步（Colab 已經內建 PyTorch；沒裝的話這格會跳過）。會用到工具三的 `x`、`y`。第一行是 PyTorch 的 $w$、$b$ 和最後一次算的損失，第二行是你的 `fit_logistic`：

In [ ]:
X = x.reshape(-1, 1)                                             # (40, 1)：40 天、1 個特徵
try:
    import torch
    import torch.nn.functional as F

    Xt, yt = torch.tensor(X), torch.tensor(y, dtype=torch.float64)
    w_t = torch.zeros(1, dtype=torch.float64, requires_grad=True)
    b_t = torch.zeros(1, dtype=torch.float64, requires_grad=True)
    opt = torch.optim.SGD([w_t, b_t], lr=1.0)                    # 每一步 θ ← θ − 1.0 × 梯度
    for step in range(300):
        opt.zero_grad()                                          # 把上一步的梯度清掉
        loss_t = F.binary_cross_entropy_with_logits(Xt @ w_t + b_t, yt)   # 直接吃分數 z
        loss_t.backward()                                        # 倒著算回來（EP05）：梯度存進 .grad
        opt.step()                                               # 往負梯度走一步
    w_np, b_np, losses = fit_logistic(X, y, lr=1.0, steps=300)   # 你在第 6 節寫的 fit_logistic
    print(w_t.item(), b_t.item(), loss_t.item())
    print(w_np[0], b_np, losses[299])
except ImportError:
    print("沒有安裝 PyTorch：跳過這格")
except NotImplementedError as e:
    print(f"{e}：先完成第 6 節的動手做，或執行最後的參考解答，再回來執行這格")

- 300 步之後只差最後一兩位（捨入誤差）。`loss_t` 是走第 300 步之前算的損失，所以對 `losses[299]`。
- PyTorch 的訓練迴圈也是四步：算預測和損失 → `backward()` 算梯度 → `step()` 走一步。它會把梯度**加**到 `.grad` 上（EP09 會看到為什麼），所以每一步先 `zero_grad()`。
- SGD 是**隨機梯度下降 stochastic gradient descent**：每一步只用隨機抽的一小批資料（EP12）；這裡每一步都用全部 40 天，就是全批次梯度下降。`torch.optim` 裡這些負責「走一步」的叫優化器 optimizer（EP15）。
- `F.binary_cross_entropy_with_logits` 直接吃分數 $z$；`F.binary_cross_entropy` 吃機率 $p$，$p$ 存成剛好 1 時它把 $\ln$ 截在 −100，算出 100（正確是 1000），梯度是 0，學不到。

## 7. 弄壞它

每一個都先**猜**結果，再執行，最後用自己的話解釋。完整的解釋在網站的「弄壞它」段落。

這一節用到工具三的 `x`、`y`、`temp`，和第 6 節寫的 `gradient_descent`、`bce_loss`、`bce_grad`、`fit_logistic`、`accuracy`、`make_weather_data`、`sigmoid`（網站上是 `from ep08.gd import ...`）；還沒寫完的話，每一格只會印出提示。重新啟動過 kernel 的話，先從第一格依序執行到第 6 節的檢查。弄壞它 4、5 有幾格要走 10 萬步，各要跑幾秒。

### 弄壞它 1：學習率太小、太大

工具二在碗上看過 0.34 發散。換成邏輯迴歸，四個學習率各走 300 步。每一行是學習率，和第 1、2、3、10、100、298、299、300 步的損失：

In [ ]:
X = x.reshape(-1, 1)                           # (40, 1)：40 天、1 個特徵（同 PyTorch 那一段）
try:
    for lr in [0.01, 1, 10, 100]:
        w_, b_, losses = fit_logistic(X, y, lr, 300)
        print(lr, np.round(losses[[1, 2, 3, 10, 100, 298, 299, 300]], 4))
except NotImplementedError as e:
    print(f"{e}：先完成第 6 節的動手做，或執行最後的參考解答，再回來執行這格")

0.01 太小：300 步只降到 0.444。10 比 1 快，100 步就到谷底 0.1845。照工具二，上限是 2 ÷ 最大的彎曲程度；起點每天 $p = 0.5$，sigmoid 最陡（斜率 0.25），彎曲程度最多約 0.25（怎麼算見弄壞它 4），上限約 8。10 為什麼還能走？越接近谷底，每天的 $p$ 越靠近 0 或 1，sigmoid 越平，彎曲程度跟著變小：谷底附近只剩約 0.06，上限約 33（實測 33 收斂、34 不會）。

100 太大：第 25 步降到 0.22 左右，又被彈開，最後在 0.5685 和 0.5849 之間來回跳，比第 1 步還糟。它不像碗的 0.34 越飛越遠，因為邏輯迴歸的梯度有上限（$|p - y| \le 1$），每步最多走那麼遠。**學習率沒有萬用的數字，要看損失曲線試**：一開始就往上、或上下跳，先調小。

### 弄壞它 2：正負號寫反

往梯度的方向走（上坡）。第一行是第 1、10、100、300 步的損失；第二行是最後的 $(w, b)$ 和準確率：

In [ ]:
def grad_wb(theta, X_=X, y_=y):              # θ = (w, b) 排成一排；預設用 40 天的資料
    dw, db = bce_grad(theta[:1], theta[1], X_, y_)
    return np.append(dw, db)

try:
    theta, hist = gradient_descent(lambda t: -grad_wb(t), np.zeros(2), 1.0, 300)   # 減掉「負的梯度」
    print([round(bce_loss(t[:1], t[1], X, y), 4) for t in hist[[1, 10, 100, 300]]])
    print(np.round(theta, 3), accuracy(theta[:1], theta[1], X, y))
except NotImplementedError as e:
    print(f"{e}：先完成第 6 節的動手做，或執行最後的參考解答，再回來執行這格")

往上坡走，損失一路變大，沒有上限（$w$ 每步大約多 0.77）。準確率 0.1：模型學成「比 19.5 度熱就是冷」，19 度以下、25 度以上全部猜反，只有 20 到 24 度那 4 天「冷」碰巧猜對（0.1 = 1 − 0.9 只是巧合）。正負號寫反不會報錯，所以**訓練一開始，先確認損失在下降**。

### 弄壞它 3：梯度用加總、不用平均

40 天加起來，等於平均乘 40。再用同樣的規則產生 400 天，平均和加總各走 300 步（會用到弄壞它 2 的 `grad_wb`）。第一行是 40 天用加總，第 1、2、3、100、299、300 步的損失；後兩行是 400 天用平均（1）和加總（400）：

In [ ]:
try:
    theta, hist = gradient_descent(lambda t: len(y) * grad_wb(t), np.zeros(2), 1.0, 300)   # 加總＝平均 × N
    print([round(bce_loss(t[:1], t[1], X, y), 4) for t in hist[[1, 2, 3, 100, 299, 300]]])
    temp400, y400 = make_weather_data(n=400)       # 同樣的規則，400 天
    X400 = ((temp400 - 20) / 10).reshape(-1, 1)
    for scale in [1, 400]:                         # 平均、加總
        theta, hist = gradient_descent(lambda t: scale * grad_wb(t, X400, y400), np.zeros(2), 1.0, 300)
        print(scale, [round(bce_loss(t[:1], t[1], X400, y400), 4) for t in hist[[1, 2, 3, 100, 299, 300]]])
except NotImplementedError as e:
    print(f"{e}：先完成第 6 節的動手做，或執行最後的參考解答，再回來執行這格")

加總＝平均 × $N$，等於把學習率乘上 $N$。40 天是學習率 40，超過谷底附近的上限（約 33，弄壞它 1）：第 60 步到過谷底 0.1845，又被彈開，最後在 0.224 和 0.228 之間來回跳。400 天是 400，損失在 0.53 到 10.7 之間亂跳，大多比什麼都沒學的 0.693 還糟；平均的版本用同一個學習率 1，400 天也穩穩降到 0.1536。**資料一多就壞掉，是用加總才會有的問題**；LLM 的損失也是平均（EP07：除以 token 數）。

### 弄壞它 4：不標準化

直接用 5 到 35 度當特徵。先看一開始兩個方向的彎曲程度（為什麼這樣算，見下面的解釋），再用學習率 1 和 0.01 試。第一行左邊是不標準化、右邊是標準化時 $w$ 方向的彎曲程度；後三列依序是學習率、步數、前三步的損失、最後的損失、分界氣溫（特徵就是氣溫，分界是 $-\frac{b}{w}$）、準確率（最後一列走 10 萬步，要跑幾秒）：

In [ ]:
X_raw = temp.reshape(-1, 1).astype(float)      # 直接用氣溫
print(0.25 * np.mean(temp ** 2), 0.25 * np.mean(x ** 2))
try:
    for lr, steps in [(1.0, 300), (0.01, 300), (0.01, 100_000)]:
        w_, b_, losses = fit_logistic(X_raw, y, lr, steps)
        print(lr, steps, np.round(losses[[1, 2, 3]], 4), round(losses[-1], 4), round(-b_ / w_[0], 2), accuracy(w_, b_, X_raw, y))
except NotImplementedError as e:
    print(f"{e}：先完成第 6 節的動手做，或執行最後的參考解答，再回來執行這格")

第二列的 57.8517 是參數大步亂跳的結果，捨入誤差會被放大，小數點後幾位每台電腦、每種 sigmoid 寫法都可能不一樣，看整數就好。

一開始每天 $p = 0.5$，sigmoid 的斜率是 0.25（這一段的 $x$ 指特徵，不標準化時就是氣溫）：$w$ 動 $\Delta w$，$p$ 大約動 $0.25\,x\,\Delta w$（EP04 的線性近似），$w$ 的梯度 $\text{mean}\bigl((p - y)\,x\bigr)$ 就變了約 $0.25 \times \text{mean}(x^2) \times \Delta w$，所以 **$w$ 方向的彎曲程度是 $0.25 \times \text{mean}(x^2)$**，$b$ 方向是 0.25。不標準化時 $w$ 方向是 119，約是 $b$ 的 480 倍；加上兩個參數互相牽動，最窄和最長的方向，彎曲程度實際差約 2600 倍（只除以 10、不減 20，還差約 36 倍），學習率要小於 $2 \div 119 \approx 0.017$。

學習率 1 一步就飛出去（21.7、43.4、65.1）。0.01 不會飛，但 $b$ 方向每步只動一點點：300 步後分界 9.4 度、準確率 0.675，要 10 萬步才到 21.38 度。標準化後，一開始兩個方向是 0.217 和 0.25，差不多圓。**平移到 0 附近（減 20）、縮放到差不多的範圍（除以 10），兩個都要做**，同一個學習率才能讓每個方向都走得動（神經網路裡是 EP14 的正規化層 normalization layer）。

畫出不標準化的損失地形，和學習率 0.01 從 $(0, 0)$ 出發的路徑（星號是谷底；這格也要走 10 萬步）：

In [ ]:
try:
    hist_raw = gradient_descent(lambda t: grad_wb(t, X_raw, y), np.zeros(2), 0.01, 100_000)[1]   # 同上一格：0.01、10 萬步
except NotImplementedError as e:
    print(f"{e}：先完成第 6 節的動手做，或執行最後的參考解答，再回來執行這格")
else:
    WU, BU = np.meshgrid(np.linspace(-1.1, 0.3, 281), np.linspace(-3, 17, 201))
    ZU = WU[..., None] * temp + BU[..., None]                  # 特徵直接用氣溫
    LU = np.mean(np.logaddexp(0, ZU) - y * ZU, axis=-1)
    fig, ax = plt.subplots(figsize=(8, 5))
    cs = ax.contour(WU, BU, LU, levels=[0.19, 0.25, 0.4, 0.6, 1, 2, 4, 8], colors="0.7", linewidths=0.9)
    ax.clabel(cs, fmt="%g", fontsize=8, colors=MUTED)
    idx = np.unique(np.r_[np.arange(0, 20), np.geomspace(20, 100_000, 80).astype(int)])   # 前 20 步每步都畫，之後挑 80 步
    ax.plot(hist_raw[idx, 0], hist_raw[idx, 1], ".-", color=SERIES[0], markersize=4, linewidth=1.5,
            label="lr 0.01 from (0, 0)")
    for k, off in [(300, (10, -4)), (100_000, (-120, 10))]:
        ax.plot([hist_raw[k, 0]], [hist_raw[k, 1]], "o", color=SERIES[1], markersize=8, zorder=5)
        ax.annotate(f"step {k:,}: boundary {-hist_raw[k, 1] / hist_raw[k, 0]:.2f} °C", hist_raw[k], xytext=off,
                    textcoords="offset points", color=INK, fontsize=9)
    w_s, b_s = -6.708, 1.012                     # 標準化時的谷底，換回氣溫：z = (w_s / 10) × 氣溫 + (b_s − 2 w_s)
    ax.plot([w_s / 10], [b_s - 2 * w_s], "*", color=INK, markersize=14, zorder=5)
    ax.annotate("bottom", (w_s / 10, b_s - 2 * w_s), xytext=(-14, 10), textcoords="offset points", color=INK,
                fontsize=9)
    ax.set(xlim=(-1.1, 0.3), ylim=(-3, 17), xlabel="w (feature = raw temperature)", ylabel="b",
           title="without standardization: a long, narrow, tilted valley")
    tidy(ax, grid=False)
    ax.legend(loc="upper right", frameon=True, framealpha=0.9)
    plt.show()

又長又窄、斜著放的谷：學習率 0.01 沿著谷慢慢爬，第 300 步分界 9.40 度，第 100,000 步 21.38 度，還沒到谷底。

### 弄壞它 5：資料完全分得開

把規則改成「低於 20 度一定寫冷」（20 度算熱），沒有重疊。走很多步，每一行是步數、損失、$w$、$b$，和 19 度、20 度「冷」的機率（最後一行走 10 萬步，要跑幾秒）：

In [ ]:
y_sep = (temp < 20).astype(int)                # 沒有隨機誤差：冷熱完全分得開
try:
    for steps in [10, 100, 1000, 10_000, 100_000]:
        w_, b_, losses = fit_logistic(X, y_sep, 1.0, steps)
        print(steps, round(losses[-1], 4), round(w_[0], 2), round(b_, 2), np.round(sigmoid(w_[0] * np.array([-0.1, 0.0]) + b_), 4))
except NotImplementedError as e:
    print(f"{e}：先完成第 6 節的動手做，或執行最後的參考解答，再回來執行這格")

損失一直往 0 靠近，卻停不下來：步數每多 10 倍，$w$ 大約變 2 倍多。1000 步時 19 度「冷」的機率剛超過 0.5，40 天全部判對；可是 S 形再陡一點，每一天的機率都更靠近 0 或 1，損失就再小一點，梯度永遠不會是 0：**谷底在無限遠**（還是凸的，只是沒有底）。19 度被推到 0.93、20 度被推到 0.036，還會繼續推：對「隔一度就完全不一樣」越來越有把握，這叫**過度自信 overconfidence**，真實資料只要有一天例外就會非常驚訝（EP07）。防止的方法之一（例如懲罰太大的 $w$）是正則化（EP16）。

畫出來：左邊是 $w$ 的大小隨步數的變化（橫軸是對數刻度），右邊是第 10、100、1000、10,000、100,000 步的 S 形曲線（這格也要走 10 萬步）：

In [ ]:
try:
    hist_sep = gradient_descent(lambda t: grad_wb(t, X, y_sep), np.zeros(2), 1.0, 100_000)[1]   # 學習率 1、10 萬步
except NotImplementedError as e:
    print(f"{e}：先完成第 6 節的動手做，或執行最後的參考解答，再回來執行這格")
else:
    marks = [10, 100, 1000, 10_000, 100_000]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))
    shown = np.unique(np.geomspace(1, 100_000, 300).astype(int))
    ax1.semilogx(shown, -hist_sep[shown, 0], color=SERIES[0], linewidth=2)
    for k in marks:
        ax1.plot([k], [-hist_sep[k, 0]], "o", color=SERIES[0], markersize=7)
        ax1.annotate(f"w = {hist_sep[k, 0]:.2f}", (k, -hist_sep[k, 0]), xytext=(-24, 8), textcoords="offset points",
                     fontsize=9, color=INK)
    ax1.set(xlabel="step (log scale)", ylabel="|w| (steepness of the S curve)", ylim=(0, 68),
            title="perfectly separable data: w never stops growing")
    tidy(ax1)
    t_grid = np.linspace(3, 37, 681)
    for k, shade in zip(marks, ["#cde2fb", "#9ec5f4", "#5598e7", "#256abf", "#0d366b"]):   # 由淺到深
        w_k, b_k = hist_sep[k]
        ax2.plot(t_grid, sigmoid(w_k * (t_grid - 20) / 10 + b_k), color=shade, linewidth=2.2, label=f"step {k:,}")
    ax2.scatter(temp, y_sep, s=36, color=INK, alpha=0.35, zorder=3, label="data: cold if below 20 °C")
    ax2.set(xlim=(3, 37), ylim=(-0.08, 1.08), xlabel="temperature (°C)", ylabel="P(cold)",
            title="the S curve keeps getting steeper")
    tidy(ax2)
    ax2.legend(loc="upper right", bbox_to_anchor=(1.0, 0.95), frameon=True, framealpha=0.9, fontsize=8.5)
    plt.show()

$w$ 一直變大，S 形越來越陡，幾乎變成 19 到 20 度之間的階梯。

## 下一集

這集的梯度 $\text{mean}\bigl((p - y)\,x\bigr)$、$\text{mean}(p - y)$ 是我們用鏈鎖律手推的；模型只要多幾層、多幾種運算，就推不動了。下一集 **EP09 自動微分（上）：計算圖與反向傳播** 讓電腦自己算梯度：把計算記成一張圖，再倒著走回來。第 1 部「數學與工具基礎」到這集結束，第 2 部「從零打造神經網路」從下一集開始。

## 參考解答

先自己寫，卡住再看。執行下一格會用參考解答取代第 6 節的 7 個函式，並重新跑一次檢查；再下一格是 `weather_data`、`train`、`draw_loss`、`draw_fit` 的參考解答，會畫出本集重點那張訓練過程的圖。之後可以回到上面，重新執行「和 PyTorch 對照」和弄壞它那幾格。

想存成 `llm-labs/ep08/gd.py` 的話，和前幾集的參考解答一樣：最上面加 `import numpy as np`，刪掉最後一行 `check_gd()`。畫圖的參考解答存成 `ep08/plot_training.py` 時，最上面加 `from pathlib import Path`、`import numpy as np` 和 `OUT = Path(__file__).parent / "training.png"`，最後一行 `show_training_plot()` 換成這段主程式（一列兩張圖，存成 `OUT`）：

```python
if __name__ == "__main__":
    import matplotlib

    matplotlib.use("Agg")    # 只存檔、不開視窗
    import matplotlib.pyplot as plt

    temp, y = weather_data()
    x = (temp - 20) / 10                                # 標準化
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.4))
    draw_loss(ax1, x, y)
    draw_fit(ax2, temp, x, y)
    fig.savefig(OUT, dpi=120, bbox_inches="tight")
    print("saved", OUT.name)
```

In [ ]:
# 只用 NumPy 和基本的 Python。存成 llm-labs/ep08/gd.py 時，最上面加 import numpy as np，並刪掉最後一行 check_gd()。


def make_weather_data(n=40, seed=0):
    """示範資料（工具三的規則）：n 天的氣溫（°C，5 到 35 的整數）和答案 y（1 = 冷、0 = 熱）。回傳 (temp, y)。"""
    rng = np.random.default_rng(seed)
    temp = rng.integers(5, 36, size=n)                 # 5 到 35 度（36 不含）
    felt = temp + rng.uniform(-6, 6, size=n)           # 每天的體感不一樣
    y = (felt < 20).astype(int)
    return temp, y


def gradient_descent(grad_fn, theta0, lr, steps):
    """每一步 θ ← θ − lr × grad_fn(θ)，走 steps 步。回傳 (theta, history)：最後的參數，和每一步的紀錄
    （theta0 是 d 個數時形狀 (steps + 1, d)；history[k] 是走了 k 步之後，history[0] 是起點）。不能改到呼叫端的 theta0。"""
    theta = np.array(theta0, dtype=float)              # 複製一份、轉成小數
    history = [theta.copy()]
    for _ in range(steps):
        theta = theta - lr * grad_fn(theta)            # 產生新陣列，不原地修改
        history.append(theta.copy())
    return theta, np.array(history)


def sigmoid(z):
    """σ(z) = 1 / (1 + e^(−z))，z 可以是一個數或陣列。數值穩定：z = ±1000 不能有警告、inf 或 nan。"""
    z = np.asarray(z, dtype=float)
    e = np.exp(-np.abs(z))                             # 指數永遠 ≤ 0，e 在 0 到 1 之間
    return np.where(z >= 0, 1 / (1 + e), e / (1 + e))[()]   # z < 0：上下同乘 e^z；[()] 讓一個數的輸入回傳一個數


def bce_loss(w, b, X, y):
    """平均二元交叉熵（nat，float）。X：(N, d)；w：(d,)；b：一個數；y：長度 N 的 0／1。用穩定的寫法。"""
    X, y = np.asarray(X, dtype=float), np.asarray(y, dtype=float)
    z = X @ np.asarray(w, dtype=float) + b
    return float(np.mean(np.logaddexp(0, z) - y * z))   # 每一筆 ln(1 + e^z) − y z


def bce_grad(w, b, X, y):
    """bce_loss 對 w、b 的梯度 (dw, db) = (Xᵀ(p − y) / N, mean(p − y))。dw 形狀 (d,)，db 是 float。"""
    X, y = np.asarray(X, dtype=float), np.asarray(y, dtype=float)
    r = sigmoid(X @ np.asarray(w, dtype=float) + b) - y   # 每一筆的 p − y
    return X.T @ r / len(y), float(np.mean(r))


def fit_logistic(X, y, lr, steps):
    """w、b 從 0 開始，用 gradient_descent 走 steps 步。回傳 (w, b, losses)：w 形狀 (d,)、b 是 float，
    losses 長度 steps + 1（losses[k] 是走了 k 步之後的損失）。"""
    X, y = np.asarray(X, dtype=float), np.asarray(y, dtype=float)
    d = X.shape[1]

    def grad_fn(theta):                                # θ = (w₁, ⋯, w_d, b) 排成一排
        dw, db = bce_grad(theta[:d], theta[d], X, y)
        return np.append(dw, db)

    theta, history = gradient_descent(grad_fn, np.zeros(d + 1), lr, steps)
    losses = np.array([bce_loss(t[:d], t[d], X, y) for t in history])
    return theta[:d], float(theta[d]), losses


def accuracy(w, b, X, y):
    """p ≥ 0.5 判成 1、否則判成 0，回傳猜對的比例（float）。"""
    X = np.asarray(X, dtype=float)
    pred = sigmoid(X @ np.asarray(w, dtype=float) + b) >= 0.5
    return float(np.mean(pred == (np.asarray(y) == 1)))


check_gd()

In [ ]:
STEPS = 300
RATES = [0.1, 1.0, 10.0]                            # 左圖的三個學習率
SHOW = [0, 10, 300]                                 # 右圖要畫的步數（學習率 1）
INK, GRAY = "#1E2A44", "#8A8F98"                    # 文字、ln 2 的虛線


def weather_data(n=40, seed=0):
    """示範資料，和 gd.py 的 make_weather_data 一樣。回傳 (temp, y)。"""
    rng = np.random.default_rng(seed)
    temp = rng.integers(5, 36, size=n)
    felt = temp + rng.uniform(-6, 6, size=n)
    return temp, (felt < 20).astype(int)


def sigmoid_plot(z):                                # 這個檔案自己的 sigmoid（不取名 sigmoid，才不會蓋掉第 6 節的）
    return np.exp(-np.logaddexp(0, -z))             # 1 / (1 + e^(−z))，不會溢位


def train(x, y, lr, steps):
    """w、b 從 0 開始做全批次梯度下降（x 是標準化後的氣溫）；回傳每一步的 (w, b, 損失)，形狀 (steps + 1, 3)。"""
    w, b, rows = 0.0, 0.0, []
    for _ in range(steps + 1):
        z = w * x + b
        rows.append((w, b, np.mean(np.logaddexp(0, z) - y * z)))
        p = sigmoid_plot(z)
        w, b = w - lr * np.mean((p - y) * x), b - lr * np.mean(p - y)
    return np.array(rows)


def draw_loss(ax, x, y):
    """左圖：RATES 的每個學習率各走 STEPS 步的損失曲線（nat），右端寫出最後的損失；一條 ln 2 的虛線（什麼都沒學）。"""
    for lr, color, dy in zip(RATES, ["C0", "C3", "C2"], [0, 6, -6]):   # dy：右端數字上下錯開
        loss = train(x, y, lr, STEPS)[:, 2]
        ax.plot(loss, color=color, linewidth=2, label=f"learning rate {lr:g}")
        ax.annotate(f"{loss[-1]:.4f}", (STEPS, loss[-1]), xytext=(4, dy), textcoords="offset points",
                    va="center", fontsize=9, color=color)
    ax.axhline(np.log(2), color=GRAY, linestyle=":", linewidth=1)
    ax.text(STEPS / 2, np.log(2) + 0.01, "ln 2 = 0.693 (learned nothing)", ha="center", va="bottom",
            fontsize=9, color=GRAY)
    ax.set(xlim=(0, STEPS + 40), ylim=(0, 0.8), xlabel="step", ylabel="loss: mean BCE (nat)",
           title="gradient descent on the weather demo data")
    ax.grid(True, color="0.9")
    ax.legend(loc="center right")


def draw_fit(ax, temp, x, y):
    """右圖：40 天的點（縱軸 1 = 冷、0 = 熱，半透明），學習率 1 在 SHOW 那幾步的 S 形曲線（由淺到深），
    第 STEPS 步的分界氣溫畫垂直虛線、標出數字。"""
    rows = train(x, y, 1.0, STEPS)
    grid = np.linspace(3, 37, 341)
    for step, shade in zip(SHOW, ["#C9CDD4", "#7FA7D9", "#2B7FD6"]):
        w, b, _ = rows[step]
        ax.plot(grid, sigmoid_plot(w * (grid - 20) / 10 + b), color=shade, linewidth=2.5, label=f"step {step}")
    ax.scatter(temp, y, s=40, color=INK, alpha=0.35, zorder=3, label="demo data (1 = cold)")
    w, b, _ = rows[STEPS]
    cut = 20 - 10 * b / w                               # z = 0 的氣溫
    ax.axvline(cut, color="#E04E3E", linestyle="--", linewidth=1.5)
    ax.annotate(f"boundary {cut:.1f} °C", (cut, 0.5), xytext=(8, 40), textcoords="offset points",
                color="#E04E3E")
    ax.set(xlim=(3, 37), ylim=(-0.08, 1.08), xlabel="temperature (°C)", ylabel="P(cold)",
           title="logistic regression: P(cold) = sigmoid(w x + b)")
    ax.grid(True, color="0.9")
    ax.legend(loc="lower left", bbox_to_anchor=(0.0, 0.06))


show_training_plot()